# 파트 2. 프롬프트 설계

파트 1에서 두 가지를 확인했습니다. 같은 프롬프트에도 응답이 달라지고, 형식이 칸을 요구하면 모델은 원문에 없는 내용으로 칸을 채웁니다. 프롬프트 설계는 이 두 문제를 프롬프트로 줄이고, 줄어들었는지 확인하는 일입니다. 이 파트에서는 같은 회의록과 일지로 프롬프트를 고쳐 가며, 무엇을 더하면 응답의 무엇이 바뀌는지를 코드로 측정합니다.

이 파트를 마치면 다음을 할 수 있습니다.

1. 프롬프트의 다섯 요소(역할 지정, 맥락, 형식, 제약, 예시)가 응답의 어느 부분을 바꾸는지 설명하고, 요소를 하나씩 더해 업무 프롬프트를 작성합니다.
2. 허용 라벨, Few-shot 예시, 출력 형식을 정해 분류 결과의 변동을 줄입니다.
3. 결과를 사용할 곳(문서, 엑셀, 프로그램)에 맞춰 출력 형식을 지정하고, 형식이 지켜졌는지 코드로 확인합니다.
4. CoT를 적용할 작업과 적용하지 않을 작업을 구분합니다.
5. 매번 같은 규칙은 시스템 프롬프트에, 건마다 바뀌는 재료는 사용자 메시지에 둡니다.
6. 긴 문서를 한 번에 넣을지 나눌지 정하고, 한 작업을 프롬프트 체인으로 나눠 단계 사이에서 결과를 검증합니다.
7. 루브릭으로 프롬프트를 고치기 전후를 비교하고, 검증한 프롬프트를 템플릿으로 남깁니다.

예제의 실행 결과는 2026-11-26에 Claude 모델로 실행한 값이고, 학습 노트북을 실행한 출력을 그대로 실었습니다. 응답은 실행할 때마다, 그리고 모델마다 다릅니다. 여러분의 결과에서는 문장이 같은지가 아니라 같은 현상이 나타나는지를 봅니다.

## 이 파트에서 만드는 것

> **요점**
>
> 이 파트의 재료는 회의록, 현장 코멘트, 회의 받아쓰기, 교대 일지, 알람 로그이고, 결과물은 요약, 분류 결과, 액션아이템 표, 안전 항목 목록, 그리고 다시 쓸 수 있는 프롬프트 템플릿입니다. 모든 재료는 가상 데이터입니다.

프롬프트 설계는 매주 반복하는 업무를 같은 품질로 처리하는 프롬프트를 만드는 일입니다. 이 파트에서는 재료마다 업무 하나를 정해 모델에게 맡기고, 응답이 업무에 쓸 수 있는지 확인합니다.

| 재료 | 재료의 예 | 만드는 것 | 예제 |
| --- | --- | --- | --- |
| 장비팀 주간회의록(7줄) | `3호기 PM(예방정비)이 부품 수급 지연으로 이틀 밀림. 목요일로 재조정.` | 팀장 보고용 요약, 액션아이템 표, JSON, 보고서 초안 | 2-1, 2-3, 2-8, 2-9 |
| 현장 코멘트(8건과 3건) | `5호기 알람 E-201 하루 세 번째` | 설비이상, 품질, 안전, 운영으로 분류한 결과 | 2-2 |
| 회의 받아쓰기(회의에서 한 말을 그대로 옮겨 적은 녹취록, 26줄) | `(정OO) 네 이번 주 내로 하겠습니다.` | 결정과 미결의 목록, 액션아이템 표 | 2-7 |
| 교대 인수인계 일지(400줄) | `케미컬 보관장 앞 바닥에 액체 흔적, 흡착포로 처리 후 환경팀 통보` | 안전 항목 14건의 목록 | 2-6 |
| 야간 설비 알람 로그(10건) | `5. 04:12 4호기 진공펌프 온도 상한 근접 (85도 / 상한 90도)` | 조치 필요와 모니터링의 분류, 조치 필요 건의 표 | 실습 2, 3 |

마지막 결과물인 프롬프트 템플릿은 검증을 마친 프롬프트에서 매번 바뀌는 곳만 빈칸으로 남긴 문서입니다. 팀원이 다음 주에 같은 업무를 같은 방식으로 처리할 수 있게 합니다.

이 파트는 다음 순서로 진행합니다.

1. 역할 지정, 맥락, 형식, 제약의 네 요소를 하나씩 더하며 응답이 바뀌는 곳을 확인합니다. 다섯 번째 요소인 예시는 다음 절의 Few-shot에서 다룹니다.
2. 분류 결과의 변동을 줄이는 Few-shot과, 결과를 사용할 곳에 맞춘 출력 형식을 다룹니다.
3. CoT, 시스템 프롬프트, 긴 문서의 처리, 프롬프트 체인을 다룹니다. 한 번의 요청에 맡기기 어려운 일에 쓰는 방법입니다.
4. 응답을 검증하는 방법과 프롬프트를 템플릿으로 남기는 방법을 정리합니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

일자마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 일자의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data/1일차'


def read_material(name):
    """재료 폴더(data/1일차)의 파일을 읽어 문자열로 돌려줍니다."""
    return open(f'{DATA}/{name}', encoding='utf-8').read().strip()


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 프롬프트의 다섯 요소

> **요점**
>
> 프롬프트는 역할 지정, 맥락, 형식, 제약, 예시의 다섯 요소로 나눌 수 있습니다. 요소마다 응답에서 바뀌는 부분이 다르므로, 하나씩 더하며 응답의 어느 부분이 바뀌는지 확인해야 필요한 줄만 남길 수 있습니다.

프롬프트의 요소는 다섯 가지입니다. 기준이 비어 있으면 응답의 변동이 커집니다. 형식이 칸을 요구하는데 제약이 없으면 원문에 없는 내용으로 칸이 채워집니다(예제 2-1).

- **역할 지정**: 모델이 어떤 입장에서 답할지 정합니다. 예: "너는 반도체 장비팀 파트장이다".
- **맥락**: 결과를 누가 읽고 어디에 쓰는지 알려 줍니다. 예: "팀장은 설비 가동률에 영향을 주는 리스크를 가장 궁금해한다".
- **형식**: 결과의 구성을 지정합니다. 예: "핵심 3줄 요약, 결정사항 표, 액션아이템 목록".
- **제약**: 결과가 지켜야 할 조건이나 하지 말아야 할 일입니다. 예: "회의록에 없는 내용을 추가하지 마".
- **예시**: 원하는 결과를 한두 개 보여 줍니다. 다음 절의 Few-shot입니다.

Anthropic의 공식 문서는 역할을 시스템 프롬프트에 두면 모델의 행동과 말투가 그 용도에 맞춰진다고 설명하고, 한 문장만 적어도 차이가 난다고 말합니다(확인일 2026-11-23). 다만 어느 요소가 응답의 무엇을 바꾸는지는 업무와 모델에 따라 다르므로, 여러분의 업무에서 직접 확인합니다.

### 나쁜 프롬프트의 세 유형

결과가 기대와 다른 프롬프트는 대개 필요한 정보나 기준이 빠져 있습니다. 빠진 것에 따라 세 유형으로 나눕니다. 고치는 방법은 모두 빠진 것을 채우는 일입니다. 표의 프롬프트는 유형을 설명하려고 지은 예이고 실행한 것이 아닙니다.

| 유형 | 프롬프트의 예 | 문제 | 고치는 방법 |
| --- | --- | --- | --- |
| 기준을 적지 않음 | "이 코멘트들 알아서 잘 정리해줘" | 무엇을 기준으로 묶고 어떤 순서로 놓을지를 모델이 정합니다. 실행할 때마다 기준이 바뀔 수 있습니다 | 묶는 기준(예: 설비별)과 정렬 순서(예: 조치가 급한 순)를 직접 적습니다 |
| 모델이 알 수 없는 것을 물음 | "우리 라인의 PM 주기가 적절한지 봐줘" | 사내 사정은 모델이 학습한 글에 없습니다 | 일반 지식만 묻는 질문으로 나누고, 사내 데이터가 필요한 부분은 근거를 프롬프트에 넣거나 사람이 확인합니다 |
| 지시를 번복함 | "표로 정리해줘", 다음 질문에서 "아니, 목록으로 바꿔줘", 그다음에 "역시 표가 낫겠어" | 번복한 지시도 대화 이력에 남아 요청마다 함께 입력됩니다 | 생각을 정리한 최종본 하나만 새 채팅에서 보냅니다 |

### 요소를 하나씩 더해 보기

같은 회의록에 역할 지정, 맥락, 형식, 제약을 하나씩 더한 프롬프트 다섯 개(0~4단계)를 보냅니다. 단계마다 새 채팅에서 보내므로 앞 단계의 영향은 없습니다. 응답은 눈으로만 읽지 않고, 응답에서 다음 네 가지를 코드로 세어 비교합니다.

- 응답의 길이
- "확인 필요"가 적힌 횟수(제약을 넣으면 모델이 비어 있는 칸을 표시하는지 봅니다)
- 담당 칸에 직위(설비 엔지니어, 품질 담당, 파트장)가 적힌 행의 수. 회의록에 담당이 정해져 있는 곳은 휴가 조정의 파트장 한 곳뿐이므로, 이 수가 늘면 담당을 지어낸 것인지 의심합니다
- 회의록에 없는 숫자(요약이 날짜나 수치를 덧붙였는지 봅니다)

**예제 2-1. 프롬프트 요소를 하나씩 더하기**

In [ ]:
# 예제 2-1
import re

minutes = read_material('회의록_8월1주차.txt')
print(minutes)

role = '너는 반도체 장비팀 파트장이다.\n'
context = ('내일 아침 팀장 보고에 쓸 자료다.\n'
           '팀장은 설비 가동률에 영향을 주는 리스크를 가장 궁금해한다.\n')
form = ('형식:\n- 핵심 3줄 요약\n- 결정사항 표 (안건 | 결정 | 비고)\n'
        '- 액션아이템 목록 (담당자와 기한 포함)\n\n')
constraints = ("제약:\n- 회의록에 없는 내용을 추가하지 마\n"
               "- 담당자나 기한이 불명확하면 '확인 필요'로 표기해\n"
               "- 존댓말 보고체로 써줘\n\n")

# 같은 회의록에 요소를 하나씩 더한 프롬프트 다섯 개
prompts = {
    '0단계 지시만': '다음 회의록을 요약해줘.\n\n' + minutes,
    '1단계 + 역할 지정': role + '팀장 보고용으로 다음 회의록을 요약해줘.\n\n' + minutes,
    '2단계 + 맥락': role + context + '다음 회의록을 요약해줘.\n\n' + minutes,
    '3단계 + 형식': role + context + '다음 회의록을 요약해줘.\n\n' + form + minutes,
    '4단계 + 제약': role + context + '다음 회의록을 요약해줘.\n\n' + form + constraints + minutes,
}

[장비팀 주간회의록 - 8월 1주차]
참석: 파트장 1, 설비 엔지니어 4, 품질 담당 1
1. 3호기 PM(예방정비)이 부품 수급 지연으로 이틀 밀림. 목요일로 재조정.
2. 2라인 파티클 카운트가 지난주 대비 15% 상승. 원인 미상, 필터 교체 예정.
3. 신입 2명 클린룸 교육 수요일 완료 예정. 교육 후 야간조 투입.
4. 품질팀 요청: 지난달 스크랩 사유 코드 정리해서 금요일까지 회신.
5. 여름 휴가 일정 이번 주 중 확정. 겹치는 조는 파트장이 조정.
6. 4호기 진공펌프 소음 보고 2건. 다음 PM 때 베어링 점검 추가.
7. 안전: 케이블 트레이 정리 미흡 지적(안전점검). 이번 주 내 재정리.


단계마다 새 채팅에서 보냅니다.

In [ ]:
# 예제 2-1 (단계마다 새 채팅에서 보냅니다)
summaries = {}
for stage, prompt in prompts.items():
    print(f'===== {stage} =====')
    chat = llm.Chat()
    chat.say(prompt)
    summaries[stage] = chat.last.text
    print()

===== 0단계 지시만 =====
**장비팀 주간회의록 요약 (8월 1주차)**

참석: 파트장 1명, 설비 엔지니어 4명, 품질 담당 1명

**설비 이슈**
- **3호기 PM**: 부품 수급 지연으로 이틀 밀려 목요일로 재조정
- **2라인 파티클**: 지난주 대비 15% 상승, 원인 미상. 우선 필터 교체 예정
- **4호기 진공펌프**: 소음 보고 2건. 다음 PM 때 베어링 점검 추가

**인력/운영**
- 신입 2명 클린룸 교육을 수요일에 마치고, 이후 야간조에 투입
- 여름 휴가 일정은 이번 주 중 확정. 조별로 겹치면 파트장이 조정

**대외 요청/안전**
- 품질팀 요청: 지난달 스크랩 사유 코드를 정리해 금요일까지 회신
- 안전점검에서 케이블 트레이 정리 미흡 지적. 이번 주 내 재정리

**주요 일정**
- 수요일: 신입 클린룸 교육 완료
- 목요일: 3호기 PM
- 금요일: 스크랩 사유 코드 회신 마감
- 이번 주 내: 휴가 일정 확정, 케이블 트레이 재정리, 2라인 필터 교체(일정 미정)

참고로 2라인 파티클 상승은 원인이 불명확해서, 필터 교체 후에도 수치를 계속 확인해야 할 것 같습니다.

===== 1단계 + 역할 지정 =====
**[장비팀 주간회의록 요약 – 8월 1주차]**

**1. 주요 이슈 (팀장님 확인 필요)**
- **2라인 파티클 증가**: 지난주 대비 15% 상승했고, 원인은 아직 파악되지 않았습니다. 우선 필터를 교체하고 이후 추이를 모니터링하겠습니다.
- **3호기 PM 지연**: 부품 수급이 늦어져 일정이 2일 밀렸고, 목요일로 재조정했습니다.
- **4호기 진공펌프 소음**: 소음 보고가 2건 접수되었습니다. 다음 PM 때 베어링 점검을 추가하겠습니다.

**2. 진행 및 조치 사항**

| 항목 | 내용 | 기한 |
|---|---|---|
| 3호기 PM | 목요일로 재조정 | 목 |
| 2라인 필터 교체 | 파티클 상승 대응 | 금주 중 |
| 신입 2명 교육 | 클린룸 교육 완료 후 야간조 투입

응답을 코드로 세어 단계를 비교합니다.

In [ ]:
# 예제 2-1 (응답을 코드로 세어 비교합니다)
def numbers_in(text):
    return set(re.findall(r'\d+', text))


def assigned_rows(text):
    """'담당' 칸이 있는 표에서 직위가 적힌 행의 수를 셉니다."""
    roles = ('설비 엔지니어', '품질 담당', '파트장')
    lines = text.splitlines()
    count, index = 0, 0
    while index < len(lines):
        is_header = (lines[index].startswith('|') and '담당' in lines[index]
                     and index + 1 < len(lines) and set(lines[index + 1]) <= set('|-: '))
        if not is_header:
            index += 1
            continue
        index += 2                                          # 머리글 다음 줄은 구분선이므로 두 줄을 건너뜁니다
        while index < len(lines) and lines[index].startswith('|'):
            count += any(role in lines[index] for role in roles)
            index += 1
    return count


for stage, text in summaries.items():
    extra = sorted(numbers_in(text) - numbers_in(minutes), key=int)
    print(f'{stage}: {len(text):,}자, "확인 필요" {text.count("확인 필요")}번, '
          f'담당 칸에 직위가 적힌 행 {assigned_rows(text)}개, 회의록에 없는 숫자 {extra}')

0단계 지시만: 558자, "확인 필요" 0번, 담당 칸에 직위가 적힌 행 0개, 회의록에 없는 숫자 []
1단계 + 역할 지정: 737자, "확인 필요" 1번, 담당 칸에 직위가 적힌 행 0개, 회의록에 없는 숫자 []
2단계 + 맥락: 922자, "확인 필요" 1번, 담당 칸에 직위가 적힌 행 0개, 회의록에 없는 숫자 []
3단계 + 형식: 1,478자, "확인 필요" 0번, 담당 칸에 직위가 적힌 행 7개, 회의록에 없는 숫자 []
4단계 + 제약: 1,323자, "확인 필요" 12번, 담당 칸에 직위가 적힌 행 1개, 회의록에 없는 숫자 []


0단계부터 2단계까지의 응답에는 담당 칸이 있는 표가 없습니다. 1단계와 2단계에서 센 "확인 필요" 1번은 제목("팀장님 확인 필요", "보고 전 확인 필요")에 쓰인 것이고, 비어 있는 칸을 표시한 것이 아닙니다. 3단계에서 형식이 담당 칸을 요구하자 액션아이템 표의 담당 칸이 모두 직위로 채워졌습니다(7행). 회의록에 담당이 정해져 있는 곳은 휴가 조정의 파트장 한 곳뿐인데, 신입 교육의 담당은 `파트장 / 교육 담당 (지정 필요)`, 나머지 다섯 행은 `설비 엔지니어 (지정 필요)`로 적혔습니다. 7행에서 휴가 조정을 뺀 6행이 모델이 채운 값입니다. 이번 실행에서 모델은 표 위에 담당자가 명시되지 않은 항목을 "(지정 필요)"로 표시했다고 밝혔지만, "확인 필요"는 한 번도 쓰지 않았습니다. 같은 뜻의 다른 표기(`지정 필요`)는 "확인 필요"를 세는 코드로는 찾지 못합니다. 4단계에서 제약을 더하자 "확인 필요"가 12번 나왔고, 담당 칸에 직위가 적힌 행은 휴가 조정(파트장) 1행으로 줄었습니다.

맥락의 효과는 2단계 응답에서 보입니다. 안건 일곱 개 가운데 2라인 파티클, 3호기 PM, 4호기 진공펌프를 "가동률 리스크"로 묶어 맨 앞에 두고, 영향의 크기(높음, 중간, 중간)를 적었습니다. 0단계 응답은 주제별로 묶었습니다. 1단계 응답은 세 설비 이슈를 "주요 이슈"로 앞에 두었지만 가동률을 기준으로 순위를 매기지는 않았습니다. 다만 2단계 표에 적힌 "추가 정지나 수율 영향 가능", "돌발 정지 가능성 증가", "비계획 다운타임 발생"과 "초기에는 숙련도 면에서 모니터링이 필요합니다"는 회의록에 없는 모델의 추측과 제안입니다. 3단계 결정사항의 비고에도 "PM 전 고장 시 비계획 정지 가능성" 같은 문장이 있습니다. 맥락은 읽는 사람에게 맞춘 해석을 요구하므로, 그 해석이 회의록에 없는 내용으로 채워질 수 있습니다.

응답의 길이는 558자, 737자, 922자, 1,478자, 1,323자로 늘다가 4단계에서 3단계보다 155자 짧아졌습니다. 3단계 비고에 있던 추측 문장이 4단계 응답에는 없습니다. 회의록에 없는 숫자는 다섯 단계 모두 없었습니다. 숫자 비교는 `\d+`로 하므로 "이틀"을 "2일"로 바꾼 표기는 잡지 못하고, 담당을 채운 것도 잡지 못합니다. 무엇을 대조할지는 그 업무의 형식이 어떤 칸을 채우라고 요구하는지를 보고 정합니다. 코드로 센 값은 확인할 곳을 알려 줄 뿐이고, 판정은 회의록과 대조한 사람이 합니다.

### 요소를 더해도 필요한 확인

요소를 더하면 응답이 읽는 사람에게 맞춰지지만, 요소를 더한다고 응답이 원문과 일치한다는 보장은 없습니다. 예제 2-1의 3단계에서는 형식이 담당 칸을 요구하자 회의록에 없는 담당이 채워졌고, 제약을 더하자 줄었습니다. 파트 1에서 본 칸 채우기와 같은 현상입니다. 제약은 이 현상을 줄이는 수단이고, 줄었는지는 응답의 담당과 기한 칸을 회의록과 대조해 확인합니다.

요소를 하나씩 더하는 이유는 필요한 줄만 남기기 위해서입니다. 한꺼번에 다 넣으면 응답이 좋아졌을 때 어느 줄 덕분인지 알 수 없고, 프롬프트가 길어져 관리하기도 어렵습니다. 고친 프롬프트가 정말 나아졌는지 점수로 비교하는 방법은 "검증" 절의 루브릭에서 다룹니다.

## Few-shot으로 분류 고정하기

> **요점**
>
> 분류에서는 허용 라벨로 분류 이름을 고정합니다. 실행마다 라벨이 달라지는 입력은 예시로 라벨을 정해 줍니다. 출력 형식은 설명만 하지 않고 실제 출력 줄의 예를 보여 줘야 코드가 응답을 읽을 수 있습니다. 모델이 적은 확신도는 측정값이 아니라 생성한 글이므로 사람이 먼저 볼 줄을 고르는 참고로만 씁니다.

Few-shot은 프롬프트에 입력과 정답의 예시를 몇 개 넣어 규칙을 보여 주는 방법이고, 예시가 없는 것을 zero-shot이라고 합니다. 모델은 프롬프트의 글에 이어질 법한 글을 생성하므로, 입력에서 정답으로 이어지는 예시가 앞에 있으면 같은 패턴으로 이어 쓰기 쉽습니다. 분류, 추출, 서식 변환처럼 규칙은 분명하지만 말로 설명하기 번거로운 일에 씁니다.

분류를 요청할 때 필요한 것은 세 가지입니다.

- **허용 라벨**: 쓸 수 있는 분류 이름을 적습니다. 적지 않으면 분류 이름과 개수가 응답마다 바뀝니다.
- **예시**: 입력과 정답 라벨의 쌍을 보여 줍니다. Anthropic의 공식 문서는 예시를 3~5개 넣고, 엣지 케이스를 포함해 다양하게 고르며, 실제 업무와 비슷한 것을 고르라고 안내합니다(확인일 2026-11-23).
- **출력 형식**: 한 줄에 하나씩 `번호 | 라벨`처럼 정하면 코드가 응답을 읽을 수 있습니다.

먼저 허용 라벨 없이 같은 프롬프트를 세 번 보내 분류 이름이 어떻게 달라지는지 봅니다. 현장 코멘트 8건을 씁니다.

**예제 2-2. 허용 라벨과 예시로 분류 고정하기**

In [ ]:
# 예제 2-2
import re

comments = read_material('현장코멘트_8건.txt')
print(comments)
print('-' * 40)

# 허용 라벨 없이 같은 프롬프트를 새 채팅에서 세 번 보내, 분류 이름만 비교합니다
sort_prompt = ('다음 현장 코멘트를 항목별로 분류해줘. '
               '첫 줄에 분류 이름만 쉼표로 구분해 적고, 그 아래에 번호별로 분류를 적어줘.\n\n' + comments)
for number in [1, 2, 3]:
    chat = llm.Chat(model='소형')
    chat.say(sort_prompt, show=False)
    print(f'[{number}번째] {chat.last.text.splitlines()[0]}')

1. 3호기 로드락 도어 열림 지연 반복 발생
2. 세정 후 웨이퍼 표면 얼룩 2매 발견
3. 케미컬 보관장 앞 바닥에 액체 흔적
4. 야간조 인수인계 노트 형식 제각각
5. 5호기 알람 E-201 하루 세 번째
6. 신규 레시피 적용 후 두께 산포 커짐
7. 방진복 지퍼 파손 3벌 : 교체 필요
8. 자재 반입 카트 바퀴 소음
----------------------------------------
[1번째] 설비, 품질, 안전·환경, 운영, 자재·물류
[2번째] 설비, 품질, 안전, 운영, 공정, 자재
[3번째] 설비, 품질, 안전, 운영, 자재


허용 라벨과 예시를 붙입니다. 예시를 주는 방식을 세 가지로 나눠 같은 8건을 분류하고, 결과를 코드로 읽어 번호별로 나란히 놓습니다.

In [ ]:
# 예제 2-2 (라벨과 예시를 붙여 분류합니다)
from collections import Counter

labels = '반드시 [설비이상/품질/안전/운영] 네 라벨만 사용해줘.\n'
format_rule = '출력은 한 줄에 하나씩 "1 | 설비이상"처럼 번호와 라벨만 적고, 다른 말은 쓰지 마.\n\n'
examples_all = ('예시:\n- "1호기 밸브 응답 지연" → 설비이상\n- "출하 로트 외관 불량 1건" → 품질\n'
                '- "비상구 앞 자재 적치" → 안전\n- "교대 근무표 공유 누락" → 운영\n\n')
examples_one_sided = ('예시:\n- "1호기 밸브 응답 지연" → 설비이상\n- "2호기 펌프 진동 증가" → 설비이상\n'
                      '- "4호기 센서 오차 발생" → 설비이상\n\n')
variants = {
    '라벨만': labels + format_rule,
    '라벨마다 예시 1개': labels + examples_all + format_rule,
    '설비이상 예시만 3개': labels + examples_one_sided + format_rule,
}


def classify(head, runs=3):
    """같은 프롬프트를 새 채팅에서 runs번 보내, 번호마다 나온 라벨과 횟수를 모읍니다."""
    votes = {}
    for _ in range(runs):
        chat = llm.Chat(model='소형')
        chat.say('다음 기준으로 분류해줘.\n' + head + comments, show=False)
        for number, label in re.findall(r'^\s*(\d+)\s*\|\s*([^\s|]+)', chat.last.text, flags=re.M):
            votes.setdefault(number, []).append(label)
    return {number: '/'.join(f'{label}{count}' for label, count in Counter(found).most_common())
            for number, found in votes.items()}


print(pd.DataFrame({name: classify(head) for name, head in variants.items()}))

         라벨만 라벨마다 예시 1개 설비이상 예시만 3개
1      설비이상3      설비이상3       설비이상3
2        품질3        품질3         품질3
3        안전3        안전3         안전3
4        운영3        운영3         운영3
5      설비이상3      설비이상3       설비이상3
6        품질3        품질3         품질3
7        안전3        안전3         안전3
8  설비이상2/운영1      설비이상3       설비이상3


출력 형식을 설명으로만 적은 경우와 실제 줄을 보여 준 경우를 비교합니다. 세 번씩 보내, 코드가 8건을 읽은 횟수를 셉니다.

In [ ]:
# 예제 2-2 (출력 형식을 설명만 한 경우와 줄을 보여 준 경우)
formats = {'설명만': '출력은 한 줄에 하나씩 "번호 | 라벨" 형식으로만 적어줘.\n\n',
           '줄을 보여 줌': format_rule}
for name, rule in formats.items():
    readable = 0
    for _ in range(3):
        chat = llm.Chat(model='소형')
        chat.say('다음 기준으로 분류해줘.\n' + labels + examples_all + rule + comments, show=False)
        rows = re.findall(r'^\s*(\d+)\s*\|\s*([^\s|]+)', chat.last.text, flags=re.M)
        readable += len(rows) == 8
    print(f'{name}: 세 번 가운데 코드가 8건을 읽은 횟수 {readable}, 마지막 응답의 첫 줄 {chat.last.text.splitlines()[0]!r}')

설명만: 세 번 가운데 코드가 8건을 읽은 횟수 2, 마지막 응답의 첫 줄 '1. 3호기 로드락 도어 열림 지연 반복 발생 | 설비이상'
줄을 보여 줌: 세 번 가운데 코드가 8건을 읽은 횟수 3, 마지막 응답의 첫 줄 '1 | 설비이상'


같은 대화에 새 코멘트 3건을 넣어, 규칙이 대화 안에서 유지되는지 봅니다.

In [ ]:
# 예제 2-2 (같은 대화에 새 코멘트 3건을 넣습니다)
chat = llm.Chat(model='소형')
chat.say('다음 기준으로 분류해줘.\n' + variants['라벨마다 예시 1개'] + comments, show=False)
chat.say('다음 3건도 같은 기준으로 분류해줘.\n' + read_material('현장코멘트_추가_3건.txt'))

9 | 설비이상
10 | 품질
11 | 안전


확신도(confidence)는 모델이 자기 분류를 얼마나 확신하는지 상, 중, 하로 적게 한 값입니다. 이유와 확신도를 붙여 같은 프롬프트를 세 번 보내고, 번호별로 라벨과 확신도가 어떻게 달라지는지 봅니다.

In [ ]:
# 예제 2-2 (이유와 확신도를 붙여 세 번 보냅니다)
confidence_prompt = ('다음 현장 코멘트 8건을 분류해줘. 라벨은 [설비이상/품질/안전/운영] 가운데 하나만 써.\n'
                     '출력은 "번호 | 라벨 | 이유 한 줄 | 확신도(상/중/하)" 형식의 줄로만 적어줘.\n\n' + comments)
runs = []
for number in [1, 2, 3]:
    chat = llm.Chat(model='소형')
    chat.say(confidence_prompt, show=False)
    rows = re.findall(r'^\s*(\d+)\s*\|\s*([^|]+?)\s*\|[^|]*\|\s*([상중하])', chat.last.text, flags=re.M)
    runs.append({int(row_number): f'{label}/{level}' for row_number, label, level in rows})
print(pd.DataFrame(runs, index=['1회', '2회', '3회']).T)

       1회      2회      3회
1  설비이상/상  설비이상/상  설비이상/상
2    품질/상    품질/상    품질/상
3    안전/상    안전/상    안전/중
4    운영/상    운영/상    운영/상
5  설비이상/상  설비이상/상  설비이상/상
6    품질/상    품질/중    품질/상
7    안전/중    안전/상    안전/중
8  설비이상/하    운영/하  설비이상/하


허용 라벨이 없을 때는 분류 이름과 개수가 실행마다 달랐습니다. 첫 번째는 5개였고 안전과 환경, 자재와 물류를 각각 한 이름으로 묶었습니다. 두 번째는 6개(공정이 들어감), 세 번째는 5개(설비, 품질, 안전, 운영, 자재)였습니다. 분류 결과를 집계하거나 다른 시스템에 넣으려면 이름이 같아야 하므로 이 결과는 그대로 쓸 수 없습니다.

허용 라벨을 주자 8건이 모두 네 라벨로만 분류되었습니다. 세 방식을 세 번씩 보낸 결과에서 8번(자재 반입 카트 바퀴 소음)만 라벨이 달랐습니다. 라벨만 준 경우에는 설비이상이 두 번, 운영이 한 번이었고, 예시를 준 두 방식에서는 세 번 모두 설비이상이었습니다. 나머지 7건은 세 방식 모두 세 번 다 같은 라벨이었습니다. 예시를 주자 실행마다 라벨이 달라지던 줄이 한 라벨로 정해진 것입니다. 8번의 정답은 정하지 않았으므로 그 라벨이 맞는지는 사람이 판단합니다. 예시를 설비이상에만 준 방식에서도 다른 줄이 설비이상으로 치우치지는 않았습니다. 다만 치우칠 만한 줄이 8번 하나뿐이어서 치우침이 없다고 단정하지는 않습니다. 실행마다 라벨이 달라지는 줄은 8번처럼 기준이 애매한 줄이므로, 그런 줄과 비슷한 입력에 정답 라벨을 붙여 예시로 넣습니다.

출력 형식을 `"번호 | 라벨" 형식으로`라고 설명으로만 적은 경우에는 세 번 가운데 두 번 코드가 8건을 읽었습니다. 마지막 응답은 첫 줄이 `1. 3호기 로드락 도어 열림 지연 반복 발생 | 설비이상`처럼 원문을 덧붙인 형식이어서 코드가 8건을 읽지 못했습니다. `"1 | 설비이상"`처럼 실제 줄을 보여 주자 세 번 모두 코드가 8건을 읽었습니다. 설명만 한 경우에 형식이 어긋난 원인은 확인하지 않았습니다. 응답을 코드로 읽을 때는 읽은 줄 수가 기대와 같은지 먼저 확인합니다.

같은 대화에 새 코멘트 3건을 넣었을 때 분류는 설비이상, 품질, 안전이었고 모두 허용 라벨 안이었습니다. 이 대화에서는 앞의 기준과 예시가 대화 이력에 있고, 요청마다 이력이 함께 전송됩니다.

확신도를 붙인 세 번에서 8번의 라벨은 설비이상, 운영, 설비이상으로 달랐고, 나머지 7건의 라벨은 세 번 모두 같았습니다. 확신도는 3번(상, 상, 중), 6번(상, 중, 상), 7번(중, 상, 중)에서 실행마다 달랐습니다. 확신도도 응답 텍스트의 일부이므로 다른 글자처럼 생성됩니다. 같은 입력의 값이 바뀐다는 점이 이를 보여 줍니다. 8번은 세 번 모두 "하"였고, 앞의 비교에서 라벨이 달랐던 줄도 8번입니다. 그래서 사람이 먼저 볼 줄을 고르는 참고로는 쓸 수 있습니다. 다만 같은 줄의 확신도가 실행마다 달랐으므로, "상"이라고 적힌 줄이 맞다는 증거는 아닙니다.

## 출력 형식 지정

> **요점**
>
> 출력 형식은 결과를 사용할 곳에 맞춰 정합니다. 사람이 읽는 문서는 표와 보고서, 엑셀은 TSV, 프로그램은 JSON입니다. 형식을 지정해도 지켜졌는지는 코드로 확인하고, 지켜지지 않을 수 있다는 전제로 후속 처리를 설계합니다.

같은 회의록에서도 결과를 문서에 붙일지, 엑셀에 넣을지, 프로그램이 읽을지에 따라 필요한 형식이 다릅니다. 형식을 정하지 않으면 받은 결과를 사람이 다시 고쳐야 합니다.

| 결과를 사용할 곳 | 형식 | 프롬프트에 적을 것 |
| --- | --- | --- |
| 위키, 문서 도구 | 마크다운 표 | 열 이름, 값이 없을 때 적을 말, 표 밖의 다른 말을 쓰지 않는다는 조건 |
| 엑셀 | TSV | 열 순서, 구분자가 탭이라는 점, 머리글 줄과 코드 블록 표시를 쓰지 않는다는 조건 |
| 프로그램 | JSON | 키 이름, 값의 허용 목록, 순수 JSON만 출력한다는 조건 |
| 메일, 보고 | 보고서 초안 | 칸의 순서, 원문에 없는 내용을 쓰지 않는다는 제약 |

JSON은 이름과 값의 쌍을 중괄호로 묶어 적는 텍스트 형식이고, 프로그램이 읽을 때는 파싱합니다. 파싱은 텍스트를 해석해 프로그램이 다루는 자료 구조로 바꾸는 일입니다. 응답이 JSON 형식을 어기면 파싱에서 실패하므로, 프로그램으로 가는 결과는 형식을 지켰는지 반드시 코드로 확인합니다. 형식을 API의 기능으로 보장하는 방법이 2일차 파트 2의 구조화 출력입니다.

**예제 2-3. 같은 회의록을 네 가지 형식으로 받기**

In [ ]:
# 예제 2-3
minutes = read_material('회의록_8월1주차.txt')

# 마크다운 표
table_prompt = ('다음 회의록에서 액션아이템만 뽑아 마크다운 표로 정리해줘.\n'
                "열은 '할 일 | 담당 | 기한'. 담당이나 기한이 없으면 '확인 필요'라고 적어줘.\n"
                '표 외의 다른 말은 하지 마.\n\n' + minutes)
llm.Chat().say(table_prompt)

| 할 일 | 담당 | 기한 |
|---|---|---|
| 3호기 PM 일정 목요일로 재조정 | 확인 필요 | 목요일 |
| 2라인 필터 교체 (파티클 카운트 상승 대응) | 확인 필요 | 확인 필요 |
| 신입 2명 클린룸 교육 완료 | 확인 필요 | 수요일 |
| 신입 2명 야간조 투입 | 확인 필요 | 교육 완료 후 |
| 지난달 스크랩 사유 코드 정리 후 품질팀에 회신 | 확인 필요 | 금요일 |
| 여름 휴가 일정 확정 | 확인 필요 | 이번 주 중 |
| 휴가 일정이 겹치는 조 조정 | 파트장 | 이번 주 중 |
| 4호기 진공펌프 베어링 점검을 다음 PM 항목에 추가 | 확인 필요 | 다음 PM 때 |
| 케이블 트레이 재정리 (안전점검 지적 사항) | 확인 필요 | 이번 주 내 |


TSV로 받아 엑셀에 붙여 넣을 수 있는지 확인합니다. 세 번 보내 줄마다 탭이 두 개인지, 코드 블록 표시로 감쌌는지 셉니다.

In [ ]:
# 예제 2-3 (TSV로 받습니다. 세 번 보냅니다)
tsv_prompt = ('다음 회의록에서 액션아이템만 TSV로 정리해줘. 열은 할 일, 담당, 기한 순서이고 탭으로 구분해. '
              "담당이나 기한이 없으면 '확인 필요'라고 적어줘. 머리글 줄은 쓰지 말고, "
              '다른 말 없이 데이터 줄만 출력해.\n\n' + minutes)
for number in [1, 2, 3]:
    chat = llm.Chat(model='소형')
    chat.say(tsv_prompt, show=False)
    text = chat.last.text
    lines = text.splitlines()
    print(f'[{number}번째] {len(lines)}줄, 모든 줄에 탭이 2개: {all(line.count(chr(9)) == 2 for line in lines)}, '
          f'코드 블록으로 감쌈: {text.strip().startswith("```")}')
print(text)

[1번째] 7줄, 모든 줄에 탭이 2개: True, 코드 블록으로 감쌈: False
[2번째] 8줄, 모든 줄에 탭이 2개: True, 코드 블록으로 감쌈: False
[3번째] 9줄, 모든 줄에 탭이 2개: True, 코드 블록으로 감쌈: False
3호기 PM 목요일로 재조정	확인 필요	목요일
2라인 파티클 카운트 원인 파악 및 필터 교체	확인 필요	확인 필요
신입 2명 클린룸 교육 실시	확인 필요	수요일
교육 완료 후 신입 2명 야간조 투입	확인 필요	확인 필요
지난달 스크랩 사유 코드 정리 후 품질팀 회신	확인 필요	금요일
여름 휴가 일정 확정	확인 필요	이번 주 중
휴가 겹치는 조 조정	파트장	이번 주 중
4호기 진공펌프 소음 대응: 다음 PM 때 베어링 점검 추가	확인 필요	다음 PM
케이블 트레이 정리 재정리	확인 필요	이번 주 내


JSON으로 받아 파싱합니다. 파싱에 성공해도 허용 목록 밖의 값이 없는지까지 확인합니다.

In [ ]:
# 예제 2-3 (JSON으로 받습니다. 세 번 보냅니다)
import re

json_prompt = ('같은 회의록의 안건을 아래 JSON 배열로 정리해줘.\n'
               '[{"no": 안건번호, "topic": "안건 요약", "status": "진행|완료|예정 중 하나", '
               '"due": "기한 또는 확인 필요"}]\n'
               '다른 말 없이, 코드 블록 표시도 없이 순수 JSON만 출력해.\n\n' + minutes)
agenda_count = len(re.findall(r'^\d+\.', minutes, flags=re.M))          # 모델이 돌려준 안건 수와 비교할 기준
for number in [1, 2, 3]:
    chat = llm.Chat(model='소형')
    chat.say(json_prompt, show=False)
    try:
        items = json.loads(chat.last.text)
        outside = [item['no'] for item in items if item['status'] not in ('진행', '완료', '예정')]
        print(f'[{number}번째] 파싱 성공, 안건 {len(items)}개(회의록에는 {agenda_count}개), '
              f'허용 목록 밖의 status가 있는 안건: {outside}')
    except (json.JSONDecodeError, KeyError, TypeError) as error:
        print(f'[{number}번째] 파싱 실패: {type(error).__name__}')
print(chat.last.text)

[1번째] 파싱 성공, 안건 7개(회의록에는 7개), 허용 목록 밖의 status가 있는 안건: []
[2번째] 파싱 성공, 안건 7개(회의록에는 7개), 허용 목록 밖의 status가 있는 안건: []
[3번째] 파싱 성공, 안건 7개(회의록에는 7개), 허용 목록 밖의 status가 있는 안건: []
[{"no": 1, "topic": "3호기 PM(예방정비) 부품 수급 지연으로 이틀 밀려 목요일로 재조정", "status": "예정", "due": "목요일"}, {"no": 2, "topic": "2라인 파티클 카운트 지난주 대비 15% 상승, 원인 미상, 필터 교체 예정", "status": "예정", "due": "확인 필요"}, {"no": 3, "topic": "신입 2명 클린룸 교육 완료 후 야간조 투입", "status": "예정", "due": "수요일"}, {"no": 4, "topic": "품질팀 요청: 지난달 스크랩 사유 코드 정리 및 회신", "status": "예정", "due": "금요일"}, {"no": 5, "topic": "여름 휴가 일정 확정 및 겹치는 조 파트장 조정", "status": "진행", "due": "이번 주 중"}, {"no": 6, "topic": "4호기 진공펌프 소음 보고 2건, 다음 PM 때 베어링 점검 추가", "status": "예정", "due": "확인 필요"}, {"no": 7, "topic": "안전점검 지적: 케이블 트레이 정리 미흡, 재정리", "status": "예정", "due": "이번 주 내"}]


보고서 초안을 받습니다.

In [ ]:
# 예제 2-3 (보고서 초안으로 받습니다)
report_prompt = ('같은 회의록으로 팀장 보고서 초안을 써줘.\n'
                 '형식: 제목 한 줄 / 요약 3줄 / 이슈별 현황과 조치 / 요청사항\n'
                 "회의록에 없는 내용은 쓰지 말고, 불명확한 것은 '확인 필요'로 적어줘.\n\n" + minutes)
llm.Chat().say(report_prompt)

**8월 1주차 장비팀 주간 업무 보고**

**요약**
- 3호기 PM은 부품 수급 지연으로 2일 밀려 목요일로 재조정했고, 2라인 파티클 카운트는 전주 대비 15% 상승했으나 원인 미상으로 필터 교체를 예정 중입니다.
- 신입 2명은 수요일 클린룸 교육 완료 후 야간조에 투입 예정이며, 품질팀 요청(지난달 스크랩 사유 코드 정리)은 금요일까지 회신해야 합니다.
- 4호기 진공펌프 소음 2건과 케이블 트레이 정리 미흡(안전점검 지적)은 각각 다음 PM 베어링 점검, 이번 주 내 재정리로 조치할 예정입니다.

**이슈별 현황과 조치**

1. **3호기 PM 일정 지연**
 - 현황: 부품 수급 지연으로 PM이 2일 밀림
 - 조치: 목요일로 재조정
 - 비고: 부품 입고 일정 확인 필요

2. **2라인 파티클 카운트 상승**
 - 현황: 지난주 대비 15% 상승, 원인 미상
 - 조치: 필터 교체 예정
 - 비고: 교체 일자 및 교체 후 재측정 계획 확인 필요

3. **신입 교육 및 야간조 투입**
 - 현황: 신입 2명 클린룸 교육이 수요일 완료 예정
 - 조치: 교육 후 야간조 투입
 - 비고: 야간조 투입 시작일 확인 필요

4. **품질팀 요청: 스크랩 사유 코드 정리**
 - 현황: 지난달 스크랩 사유 코드 정리 요청 접수
 - 조치: 금요일까지 품질팀에 회신
 - 비고: 담당자 확인 필요

5. **여름 휴가 일정**
 - 현황: 이번 주 중 확정 예정
 - 조치: 겹치는 조는 파트장이 조정

6. **4호기 진공펌프 소음**
 - 현황: 소음 보고 2건
 - 조치: 다음 PM 때 베어링 점검 추가
 - 비고: 다음 PM 일자 확인 필요

7. **안전: 케이블 트레이 정리 미흡**
 - 현황: 안전점검에서 지적
 - 조치: 이번 주 내 재정리
 - 비고: 담당자 및 지적 위치 확인 필요

**요청사항**
- 회의록에 팀장님께 드릴 요청사항이 명시되어 있지 않아 **확인 필요**합니다.
- 보고 전, 위 비고의 '확인 필요' 항목 중 지원이나 

마크다운 표는 지정한 열 이름이 지켜졌고, 담당이 없는 행의 담당 칸에 "확인 필요"가 적혔습니다. 회의록의 안건 7개가 9행이 되었습니다. "신입 교육 완료"와 "야간조 투입", "휴가 확정"과 "겹치는 조 조정"을 나눠 적었기 때문이며, 액션아이템을 몇 건으로 볼지는 모델이 정하는 부분입니다.

TSV는 세 번 모두 모든 줄에 탭이 두 개였고 코드 블록으로 감싸지 않았습니다. 줄 수는 7줄, 8줄, 9줄로 실행마다 달랐습니다. 액션아이템을 몇 건으로 나눌지를 모델이 정하기 때문입니다. JSON은 세 번 모두 파싱에 성공했고, 안건이 회의록과 같은 7개였으며, `status`가 허용 목록 밖인 안건은 없었습니다. 프롬프트에서 안건의 개수를 지시하지 않았는데 개수가 맞았습니다. 보고서 초안은 지정한 칸의 순서를 지켰습니다. 이슈 일곱 개 가운데 여섯 개에 "비고" 줄을 두어 회의록에 없어 확인해야 할 정보(부품 입고 일정, 교체 일자, 담당자 등)를 나열했고, 담당자를 확인 대상으로 적은 것은 4번과 7번뿐입니다. 요청사항은 회의록에 명시되어 있지 않다며 "확인 필요"로 적었습니다.

이번 실행에서는 지시한 형식이 모두 지켜졌습니다. 그래도 코드로 확인하는 이유는 두 가지입니다. 하나는 형식이 한 번 지켜졌다고 매번 지켜지는 것은 아니라는 점입니다. TSV를 코드 블록으로 감싸거나 머리글 줄을 붙이는 응답이 나오면 엑셀에 붙여 넣을 수 없습니다. 다른 하나는 형식이 맞는 것과 값이 맞는 것은 다르다는 점입니다. `status`가 허용 목록 안에 있어도 그 값이 회의록과 맞는지는 코드가 판정하지 못합니다. 2026-11-23 실행에서는 2번 안건(2라인 파티클)의 `status`가 회의록의 "필터 교체 예정"과 달리 "진행"으로 나왔습니다. 2026-11-26 실행의 마지막 응답에서는 "예정"이었습니다. 파싱에 실패하면 어떻게 할지(재시도, 사람 확인)는 2일차에서 다룹니다.

## CoT: 중간 단계를 먼저 적게 하기

> **요점**
>
> 답을 쓰기 전에 풀이의 중간 값을 적게 하면 사람이나 코드가 그 값을 검산할 수 있습니다. 그래서 CoT는 여러 단계의 계산이나 조건 판단에 적용하고 판단이 필요 없는 일에는 적용하지 않습니다.

CoT(chain of thought)는 답에 이르는 중간 단계를 먼저 적게 하는 방법입니다. 모델은 앞의 토큰을 읽고 다음 토큰을 생성하므로, 중간 단계를 먼저 생성하면 뒤의 토큰이 그 단계를 읽고 이어집니다. 이 방법을 chain-of-thought prompting이라는 이름으로 발표한 Wei 외(2022)는 산술, 상식, 기호 조작 문제(arithmetic, commonsense, symbolic reasoning tasks)에서 정확도가 높아졌다고 보고했습니다.

그러나 이 교재가 CoT를 쓰는 이유는 정확도를 높이는 데보다 중간 값을 검산할 수 있다는 데 있습니다. 답만 받으면 맞았는지 알 방법이 없습니다. 중간 값이 한 줄씩 보이면 사람이나 코드가 그 줄을 확인할 수 있습니다.

thinking을 하는 모델은 응답을 쓰기 전에 이미 문제를 풀어 봅니다(파트 1의 "thinking을 하는 모델" 소절). 이런 모델에서도 "단계별로 보여준 뒤 답을 정리해줘"는 의미가 있습니다. thinking은 응답에 드러나지 않을 수 있지만, 이 지시는 중간 값을 응답 안에 적게 하기 때문입니다. 다만 모델의 "생각 과정"을 응답에 쓰게 하는 요청은 거절될 수 있으므로(아래 주의 상자), 생각 과정 대신 풀이나 계산 과정을 적게 합니다.

**예제 2-4. 답만 받기, 단계별로 받기, 붙이지 않을 때**

In [ ]:
# 예제 2-4
calc = ('장비 3대가 각각 다르게 돌아간다.\nA는 시간당 120장, B는 95장, C는 140장.\n'
        'A가 3시간, B가 5시간, C가 2시간 돌았다.\n총 몇 장이고, 가장 많이 만든 장비는?\n')
asks = {'답만 말해': calc + '답만 말해.',
        '단계별로': calc + '단계별로 계산 과정을 보여준 뒤 마지막에 답을 정리해줘.'}

expected_total = 120 * 3 + 95 * 5 + 140 * 2
print(f'코드로 구한 정답: 총 {expected_total:,}장, 가장 많이 만든 장비 B\n')
for name, prompt in asks.items():
    print(f'===== {name} =====')
    chat = llm.Chat(model='소형')
    chat.say(prompt)
    text = chat.last.text
    right = f'{expected_total:,}' in text or str(expected_total) in text
    print(f'정답 총합이 들어 있음: {right}, 출력 토큰 {chat.last.output_tokens}개\n')

코드로 구한 정답: 총 1,115장, 가장 많이 만든 장비 B

===== 답만 말해 =====
총 **1,115장**이고, 가장 많이 만든 장비는 **B**(475장)입니다.
정답 총합이 들어 있음: True, 출력 토큰 87개

===== 단계별로 =====
## 단계별 계산

**1단계: 장비별 생산량 계산 (시간당 생산량 × 가동 시간)**

- A: 120장/시간 × 3시간 = **360장**
- B: 95장/시간 × 5시간 = **475장**
- C: 140장/시간 × 2시간 = **280장**

**2단계: 총 생산량 계산**

360 + 475 + 280 = **1,115장**

**3단계: 가장 많이 만든 장비 비교**

360장(A), 475장(B), 280장(C) 중 가장 큰 값은 475장이므로 B입니다.

## 최종 답

- **총 생산량:** 1,115장
- **가장 많이 만든 장비:** B (475장)
정답 총합이 들어 있음: True, 출력 토큰 299개


판단이 필요 없는 일에 CoT를 적용하면 어떻게 되는지 봅니다. 회의록에서 조건에 맞는 줄을 뽑는 일을 세 번씩 보내 출력 토큰을 비교합니다.

In [ ]:
# 예제 2-4 (판단이 필요 없는 일에 붙이면)
import re

minutes = read_material('회의록_8월1주차.txt')
extract = '다음 회의록에서 "금요일"이라는 말이 들어 있는 줄의 번호만 적어줘.\n\n' + minutes
steps = '\n줄마다 "금요일"이 들어 있는지 확인한 결과를 적은 뒤, 마지막 줄에 번호만 적어줘.'
expected = {line.split('.')[0] for line in minutes.splitlines() if '금요일' in line}      # 모델의 답과 대조할 기준을 코드로 먼저 구합니다
for name, suffix in [('그냥 요청', ''), ('줄마다 확인하며 요청', steps)]:
    sizes, right = [], 0
    for number in [1, 2, 3]:
        chat = llm.Chat(model='소형')
        chat.say(extract + suffix, show=False)
        sizes.append(chat.last.output_tokens)
        last_line = (chat.last.text.strip().splitlines() or [''])[-1]
        right += set(re.findall(r'\d+', last_line)) == expected
    print(f'{name}: 출력 토큰 {sizes}, 정답({", ".join(sorted(expected))}번)만 적은 횟수 {right}/3')
print(chat.last.text)

그냥 요청: 출력 토큰 [79, 83, 66], 정답(4번)만 적은 횟수 3/3
줄마다 확인하며 요청: 출력 토큰 [203, 149, 183], 정답(4번)만 적은 횟수 3/3
1. 없음 (목요일)
2. 없음
3. 없음 (수요일)
4. 있음 (금요일)
5. 없음
6. 없음
7. 없음

4


과정을 요청하는 문구를 바꿔 같은 요청을 소형과 기본 모델에 세 번씩 보내고, 종료 이유를 봅니다.

In [ ]:
# 예제 2-4 (과정을 요청하는 문구에 따라 응답이 달라지는지 봅니다)
from collections import Counter

wordings = ['단계별로 생각하는 과정을 보여준 뒤 답을 적어줘.',
            '단계별로 판단 과정을 적은 뒤 답을 정리해줘.',
            '단계별로 풀이를 적은 뒤 답을 정리해줘.']
for model in ['소형', '기본']:
    for wording in wordings:
        reasons = Counter()
        for _ in range(3):
            chat = llm.Chat(model=model)
            chat.say(extract + '\n' + wording, show=False)
            reasons[chat.last.stop_reason] += 1
        print(f'{model} 모델, "{wording}" → 종료 이유 {dict(reasons)}')

소형 모델, "단계별로 생각하는 과정을 보여준 뒤 답을 적어줘." → 종료 이유 {'refusal': 3}
소형 모델, "단계별로 판단 과정을 적은 뒤 답을 정리해줘." → 종료 이유 {'refusal': 3}
소형 모델, "단계별로 풀이를 적은 뒤 답을 정리해줘." → 종료 이유 {'end': 3}
기본 모델, "단계별로 생각하는 과정을 보여준 뒤 답을 적어줘." → 종료 이유 {'refusal': 2, 'end': 1}
기본 모델, "단계별로 판단 과정을 적은 뒤 답을 정리해줘." → 종료 이유 {'end': 3}
기본 모델, "단계별로 풀이를 적은 뒤 답을 정리해줘." → 종료 이유 {'end': 3}


계산에서는 두 요청 모두 코드로 구한 정답(1,115장, 장비 B)과 같았습니다. 이 계산은 쉬워서 답만 말해도 맞았습니다. 차이는 응답에 중간 값이 있는지입니다. "답만 말해"의 응답은 87토큰이고 B의 475장만 있어서 총합 1,115가 어떻게 나왔는지 검산할 수 없습니다. "단계별로"의 응답은 299토큰이고 장비별 생산량(360, 475, 280)이 한 줄씩 보여 각각 검산할 수 있습니다. 이번에 "답만 말해"가 맞았다는 것을 안 것은 정답을 코드로 먼저 구했기 때문입니다.

두 번째 코드에서는 "금요일"이라는 말이 들어 있는 줄의 번호를 뽑게 했습니다. 두 요청 모두 세 번 가운데 세 번 정답(4번)만 적었습니다. 출력 토큰은 그냥 요청이 79, 83, 66토큰이고 줄마다 확인하며 요청한 쪽이 203, 149, 183토큰입니다. 평균은 약 76토큰과 178토큰으로 2.3배쯤입니다. 이 일은 응답의 번호 하나를 원문과 대조하면 충분하므로, 줄마다 적은 확인 결과는 출력 토큰만 늘립니다.

> **주의**
>
> "단계별로 생각하는 과정을 보여준 뒤"라고 요청하면 소형 모델은 세 번 모두, 기본 모델은 세 번 가운데 두 번 응답 텍스트 없이 끝났습니다. 종료 이유는 `refusal`(거절)입니다. "판단 과정을 적은 뒤"는 소형 모델에서 세 번 모두 거절되었고, 기본 모델에서는 세 번 모두 응답했습니다. 같은 문구도 모델과 실행에 따라 거절 여부가 다릅니다. 2026-11-23 실행에서는 기본 모델이 "생각하는 과정"을 세 번 모두, "판단 과정"을 세 번 가운데 한 번 거절했습니다. "풀이를 적은 뒤"는 두 모델 모두 세 번 응답했고 종료 이유가 `end`(정상 종료. Anthropic API의 `end_turn`, OpenAI 호환 API의 `stop`을 `llm.py`가 바꿔 돌려준 값)였습니다.
> 
> Anthropic의 공식 문서에 따르면 이번에 쓴 소형과 기본 모델을 포함한 최신 Claude 모델에는 요청을 거절하는 분류기가 있습니다. 모델의 생각 과정(reasoning)을 응답 본문에 쓰게 하는 프롬프트는 거절 카테고리 `reasoning_extraction`으로 거절될 수 있고, 이 카테고리의 거절은 출력이 없어도 요금이 부과됩니다(확인일 2026-11-26). 이번 실행에서는 거절 카테고리(`stop_details.category`)를 출력하지 않았으므로, 위의 거절이 이 카테고리였는지는 확인하지 않았습니다. 문서는 생각 과정 대신 짧은 설명, 결과의 근거, 수행한 작업의 요약을 요청하라고 안내합니다. 이번 실행에서 응답이 온 문구도 "풀이", "계산 과정", "줄마다 확인한 결과"를 적게 하는 것이었습니다.
> 
> OpenAI 호환 API에서는 모델과 서버마다 거절 방식이 다르고, 이 거절이 일어나지 않을 수도 있습니다. `llm.py`는 OpenAI 호환 API의 종료 이유 `content_filter`를 `refusal`로 바꿔 돌려줍니다. 응답이 비어 있으면 종료 이유를 먼저 확인합니다.

CoT를 적용할지는 다음 기준으로 정합니다.

- **적용하는 작업**: 여러 단계의 계산, 얽힌 조건의 판단, 분류의 근거가 필요한 일입니다. 중간 단계를 사람이나 코드가 검산할 수 있습니다.
- **적용하지 않는 작업**: 원문에서 뽑기, 서식 바꾸기, 번역입니다. 판단이 없어서 출력 토큰과 시간만 늘어납니다.

## 시스템 프롬프트와 멀티턴

> **요점**
>
> 매번 똑같이 적용할 역할, 어조, 출력 형식, 제약은 시스템 프롬프트에 두고, 이번 건에만 해당하는 조건과 원자료는 사용자 메시지에 둡니다. 규칙은 지켜졌는지 규칙마다 따로 확인합니다.

시스템 프롬프트는 대화 전체에 적용할 규칙을 따로 적어 두는 곳입니다. 역할, 형식, 제약을 매번 프롬프트에 다시 적으면 빠뜨리는 일이 생깁니다. 시스템 프롬프트에 한 번 적어 두면 요청마다 함께 전송됩니다. Anthropic API는 `system`이라는 별도 인자로, OpenAI 호환 API는 `messages` 맨 앞의 `system` 역할 메시지로 넣습니다. 채팅 서비스에서는 프로젝트 지침이나 사용자 지침 같은 이름의 설정란이 같은 역할을 합니다.

시스템 프롬프트와 사용자 메시지는 바뀌는 주기로 나눕니다.

- **시스템 프롬프트에 넣는 것**: 역할, 어조, 제약, 그리고 모든 응답에 똑같이 쓰는 출력 형식(예: 항상 표 또는 번호 목록)처럼 매번 똑같이 적용되는 것입니다. 결과를 사용할 곳에 따라 달라지는 형식은 사용자 메시지에 둡니다.
- **사용자 메시지에 넣는 것**: 이번 건에만 해당하는 조건과 원자료처럼 자주 바뀌는 것입니다.

멀티턴은 한 대화 안에서 질문과 응답을 여러 번 주고받는 것입니다. 모델은 앞의 대화를 기억하지 않고, 요청마다 앞의 질문과 응답 전부가 다시 입력됩니다(파트 1의 "토큰과 컨텍스트 윈도우" 절). 시스템 프롬프트도 요청마다 다시 전송되므로 대화 이력과 별개로 모든 요청에 적용됩니다. 한편 대화가 길수록 입력 토큰이 늘고, 앞의 대화 내용이 새 질문의 응답에 반영됩니다. 주제가 바뀌면 새 채팅을 엽니다.

규칙을 어기기 쉬운 질문(원자료가 없는데 담당자를 묻는 질문)을 세 가지 조건으로 보냅니다. 규칙을 시스템 프롬프트에 넣은 경우, 같은 규칙을 사용자 메시지에 넣은 경우, 규칙이 없는 경우입니다. 규칙마다 지켜졌는지 코드로 셉니다.

**예제 2-5. 시스템 프롬프트에 규칙 넣기**

In [ ]:
# 예제 2-5
import re

system_rules = '''너는 반도체 장비팀 파트장이다.

규칙:
1. 원자료에 없는 내용은 만들지 않는다
2. 불명확하면 '확인 필요'로 적는다
3. 답변은 표 또는 번호 목록으로 한다
4. 추측이 필요하면 추측이라고 밝힌다'''
question = '폐액 라벨 건 담당자가 누구야?'


def follows_rules(text):
    """규칙 2와 3을 지켰는지 확인합니다. 규칙 1과 4는 코드로 판정할 수 없어 읽어서 확인합니다."""
    return {'규칙 2 확인 필요 표기': '확인 필요' in text,
            '규칙 3 표 또는 번호 목록': bool(re.search(r'^\s*(\d+\.|\|)', text, flags=re.M))}


# 같은 규칙을 넣는 자리만 바꿔 비교합니다
conditions = {'시스템 프롬프트에 규칙': {'system': system_rules, 'message': question},
              '사용자 메시지에 같은 규칙': {'system': None, 'message': system_rules + '\n\n' + question},
              '규칙 없음': {'system': None, 'message': question}}
for name, condition in conditions.items():
    print(f'===== {name} =====')
    kept = {}
    for number in [1, 2, 3]:
        chat = llm.Chat(system=condition['system'], model='소형')
        chat.say(condition['message'], show=(number == 1))
        for rule, ok in follows_rules(chat.last.text).items():
            kept[rule] = kept.get(rule, 0) + ok
    print(f'세 번 가운데 지킨 횟수: {kept}\n')

===== 시스템 프롬프트에 규칙 =====
죄송하지만 현재 제공된 정보만으로는 폐액 라벨 건의 담당자를 확인할 수 없습니다.

| 항목 | 내용 |
|---|---|
| 요청 내용 | 폐액 라벨 건 담당자 |
| 담당자 | 확인 필요 |
| 확인 근거 | 제공된 자료 없음 |

**확인을 위해 필요한 정보**
1. 해당 폐액 라벨 건의 관련 문서나 이슈 번호
2. 해당 장비명 또는 공정명
3. 작업 요청 기록 또는 담당 배정표 여부

위 정보를 공유해 주시면 자료 기준으로 확인해 드리겠습니다. 담당자 배정표나 작업 이력 시스템이 있다면 해당 자료를 함께 전달해 주시기 바랍니다.
세 번 가운데 지킨 횟수: {'규칙 2 확인 필요 표기': 3, '규칙 3 표 또는 번호 목록': 3}

===== 사용자 메시지에 같은 규칙 =====
원자료가 제공되지 않아 담당자를 특정할 수 없습니다.

| 항목 | 내용 |
|---|---|
| 폐액 라벨 건 담당자 | 확인 필요 |
| 근거 자료 | 없음 (원자료 미제공) |

**확인에 필요한 정보**
1. 해당 건의 문서나 이슈 번호(작업지시서, 점검 기록, 메일, 메신저 기록 등)
2. 라벨 문제가 발생한 설비명 또는 폐액 종류
3. 담당 업무 분장표 또는 환경안전 관련 담당자 목록

관련 자료를 공유해 주시면 담당자를 확인해 드리겠습니다.
세 번 가운데 지킨 횟수: {'규칙 2 확인 필요 표기': 3, '규칙 3 표 또는 번호 목록': 3}

===== 규칙 없음 =====
저는 회사나 실험실의 내부 담당자 정보를 알 수 없어서 정확히 답해 드리기 어렵습니다. 아래 곳에서 확인해 보시는 것이 빠릅니다.

- **안전환경팀(EHS) 또는 환경안전 담당 부서**: 폐기물과 폐액 관리를 총괄하는 경우가 많습니다.
- **해당 실험실이나 작업장의 안전관리자**: 실험실 단위로 지정된 담당자가 있을 수 있습니다.
- **폐액 보관실이나 폐기물 보관장소에 붙은 연락처**: 게시판이나 관리 기록부에 적혀 있는 경우가 많습니다.
- 

같은 대화에서 이어서 묻고, 대화 이력에 무엇이 들어 있는지 봅니다.

In [ ]:
# 예제 2-5 (같은 대화에서 이어서 묻습니다)
follow_up = '환경팀 회신 기한은 언제야?'
for name in ['시스템 프롬프트에 규칙', '사용자 메시지에 같은 규칙']:
    condition = conditions[name]
    kept = {}
    for number in [1, 2, 3]:
        chat = llm.Chat(system=condition['system'], model='소형')
        chat.say(condition['message'], show=False)
        chat.say(follow_up, show=False)
        for rule, ok in follows_rules(chat.last.text).items():
            kept[rule] = kept.get(rule, 0) + ok
    in_history = any('규칙:' in message['content'] for message in chat.history)
    print(f'{name}: 이어서 물은 응답에서 지킨 횟수 {kept}, 대화 이력에 규칙 문장이 있음 {in_history}')

시스템 프롬프트에 규칙: 이어서 물은 응답에서 지킨 횟수 {'규칙 2 확인 필요 표기': 3, '규칙 3 표 또는 번호 목록': 3}, 대화 이력에 규칙 문장이 있음 False
사용자 메시지에 같은 규칙: 이어서 물은 응답에서 지킨 횟수 {'규칙 2 확인 필요 표기': 3, '규칙 3 표 또는 번호 목록': 3}, 대화 이력에 규칙 문장이 있음 True


규칙을 시스템 프롬프트에 넣은 세 번과 사용자 메시지에 넣은 세 번은 모두 규칙 2("확인 필요" 표기)와 규칙 3(표 또는 번호 목록)을 세 번씩 지켰고, 규칙이 없을 때는 두 규칙을 한 번도 지키지 않았습니다. 코드로 판정할 수 없는 규칙 1(원자료에 없는 내용을 만들지 않는다)과 규칙 4(추측을 밝힌다)는 응답을 읽어 확인합니다. 규칙을 넣은 두 조건에서 출력한 첫 응답은 모두 제공된 자료로는 담당자를 확인할 수 없다고 하며 담당자를 지어내지 않았습니다. 규칙이 없을 때 출력한 첫 응답도 지어내지는 않았지만, 담당일 법한 부서의 목록을 덧붙였고 "확인 필요" 표기는 없었습니다. 규칙을 넣은 응답에도 확인 방법과 필요한 정보의 목록이 덧붙었습니다. 규칙에 없는 행동은 규칙이 막지 못하므로, 막아야 하는 행동은 규칙에 적어야 합니다.

시스템 프롬프트와 사용자 메시지의 차이는 이번 실행에서 보이지 않았습니다. 두 조건의 지킨 횟수가 같았습니다. 2026-11-23 실행에서는 규칙 2를 지킨 횟수가 두 번과 세 번으로 한 번 차이 났지만, 세 번씩 보낸 결과에서 생긴 이 정도 차이는 판단 근거로 삼지 않습니다. 같은 대화에서 이어서 물은 응답에서도 두 조건 모두 두 규칙을 세 번 모두 지켰습니다. 차이가 나타난 곳은 대화 이력입니다. 사용자 메시지에 넣은 규칙은 첫 질문과 함께 대화 이력에 남습니다. `llm.Chat`은 시스템 프롬프트를 대화 이력(`chat.history`)에 넣지 않고 요청을 보낼 때마다 따로 붙입니다. 그래서 시스템 프롬프트에 두면 규칙과 건마다 바뀌는 질문이 섞이지 않고, 규칙을 한 곳에서 관리할 수 있습니다. 응답의 품질이 더 좋아진다는 근거는 이 실험에서 얻지 못했습니다. 이어서 물은 것도 한 번뿐이므로, 대화가 더 길어졌을 때 규칙이 유지되는지는 확인하지 않았습니다.

## 긴 문서 다루기

> **요점**
>
> 긴 문서는 먼저 한 번에 넣어 보고, 컨텍스트 윈도우를 넘거나 응답이 잘릴 때만 구간으로 나눕니다. 나누면 모델이 구간마다 따로 판단하므로 같은 지시와 제외 기준을 구간마다 적고, 합친 결과는 정답이나 원문과 대조합니다.

회의록 묶음, 일지, 매뉴얼처럼 긴 자료에서 필요한 항목을 뽑는 일이 있습니다. 순서는 다음과 같습니다.

1. **한 번에 넣어 봅니다.** 입력이 컨텍스트 윈도우 안에 들어가고 응답이 잘리지 않으면 나누지 않습니다. Anthropic의 공식 문서는 긴 문서를 프롬프트의 앞쪽에, 질문과 지시를 뒤쪽에 두라고 안내합니다(확인일 2026-11-23).
2. **컨텍스트 윈도우를 넘거나 응답이 잘리면 나눕니다.** 응답이 잘렸는지는 종료 이유로 확인합니다. 최대 출력 토큰에 도달해 잘리면 종료 이유가 `max_tokens`(OpenAI 호환은 `length`)입니다. 구간마다 같은 지시와 넣을 것, 뺄 것의 기준을 적어 한 구간씩 보냅니다.
3. **합치며 개수를 확인합니다.** 구간별 결과의 합과 합친 결과의 항목 수를 비교합니다.
4. **원문과 대조합니다.** 개수가 맞아도 틀린 줄은 찾아낼 수 없으므로, 넣을지 뺄지 애매한 줄은 원문에서 직접 확인합니다.

긴 문서를 구간으로 잘라 같은 지시로 하나씩 처리한 뒤 결과를 합치는 방법을 분할 처리라고 합니다. 분할 처리를 하는 이유는 컨텍스트 윈도우와 응답 길이입니다. 나누면 모델이 구간마다 따로 판단해서 같은 기준이 구간마다 다르게 적용될 수 있습니다. 4주치 교대 일지 400줄에서 안전 항목을 뽑는 일로 확인합니다. 일지에는 안전 항목 14건이 있고, 나머지는 일반 가동 기록과 안전 항목처럼 보이는 "알람 1회, 재시작 후 정상" 기록입니다. 14건의 줄 번호는 정답 파일에 있고, 응답의 표에서 줄 번호를 코드로 뽑아 정답과 대조합니다. 14건의 표가 잘리지 않도록 예제는 최대 출력 토큰을 `max_tokens=8000`으로 넉넉히 줍니다.

**예제 2-6. 400줄 일지에서 안전 항목 뽑기**

In [ ]:
# 예제 2-6
import re

diary_text = read_material('교대일지_4주.txt')
print('\n'.join(diary_text.splitlines()[:4]))
print('...')
answer_rows = read_material('교대일지_안전항목.txt').splitlines()[2:]
answer = {row.split(' | ')[0] for row in answer_rows}           # 응답의 줄 번호와 대조할 정답(14개)
content = {row.split(' | ')[0]: row.split(' | ', 2)[2] for row in diary_text.splitlines()[1:]}

safety_prompt = ('일지에서 안전 관련 항목(사람의 안전, 설비 안전장치, 화학물질·환경, 소방)을 '
                 '빠짐없이 골라 "줄 번호 | 날짜 | 내용 | 조치" 마크다운 표로 정리해줘. '
                 '표 아래 마지막 줄에 "찾은 개수: N건"을 적어줘.')


def ask_safety(log_lines, label='', instruction_first=False):
    """일지(또는 일지의 한 구간)를 보내고 (줄 번호 집합, 모델이 적은 개수, 종료 이유)를 돌려줍니다."""
    log = label + '\n'.join(log_lines)
    prompt = safety_prompt + '\n\n' + log if instruction_first else log + '\n\n' + safety_prompt
    chat = llm.Chat(model='소형', max_tokens=8000)
    chat.say(prompt, show=False)
    text = chat.last.text
    numbers = set(re.findall(r'^\|?\s*(\d{3})\s*\|', text, flags=re.M))
    claimed = re.search(r'찾은 개수:\s*(\d+)', text)
    return numbers, int(claimed.group(1)) if claimed else None, chat.last.stop_reason


def compare(found, claimed, reasons):
    wrong = found - answer
    routine = [number for number in wrong if '재시작 후 정상' in content.get(number, '')]
    return (f'표의 줄 {len(found)}개(모델이 적은 개수 {claimed}), 정답 {len(found & answer)}/14, '
            f'빠진 줄 {len(answer - found)}개, 정답이 아닌 줄 {len(wrong)}개'
            f'(그중 "재시작 후 정상" 기록 {len(routine)}개), 종료 이유 {reasons}')

[2라인 교대 인수인계 일지 · 8/4 ~ 9/1 · 가상 자료]
001 | 8/4 주간 | 세정기 A 진공도 정상 범위
002 | 8/4 주간 | 측정기 M2 파티클 측정값 기준 이내
003 | 8/4 주간 | 로드락 L1 측정 데이터 업로드 완료
...


400줄을 한 번에 넣습니다. 세 번 보냅니다.

In [ ]:
# 예제 2-6 (400줄을 한 번에 넣습니다)
for number in [1, 2, 3]:
    found, claimed, reason = ask_safety(diary_text.splitlines())
    print(f'[한 번에 {number}번째]', compare(found, claimed, [reason]))

[한 번에 1번째] 표의 줄 14개(모델이 적은 개수 14), 정답 14/14, 빠진 줄 0개, 정답이 아닌 줄 0개(그중 "재시작 후 정상" 기록 0개), 종료 이유 ['end']
[한 번에 2번째] 표의 줄 14개(모델이 적은 개수 14), 정답 14/14, 빠진 줄 0개, 정답이 아닌 줄 0개(그중 "재시작 후 정상" 기록 0개), 종료 이유 ['end']
[한 번에 3번째] 표의 줄 14개(모델이 적은 개수 14), 정답 14/14, 빠진 줄 0개, 정답이 아닌 줄 0개(그중 "재시작 후 정상" 기록 0개), 종료 이유 ['end']


100줄씩 네 구간으로 나눠 같은 지시로 보내고, 구간별 결과를 합칩니다. 세 번 보냅니다.

In [ ]:
# 예제 2-6 (네 구간으로 나눠 넣고 합칩니다)
body = diary_text.splitlines()[1:]
chunks = [body[start:start + 100] for start in range(0, 400, 100)]


def ask_chunks(instruction_first=False):
    """네 구간을 따로 보내 결과를 합칩니다. 모델이 적은 개수도 더합니다."""
    merged, claimed_total, reasons = set(), 0, []
    for index, chunk in enumerate(chunks, 1):
        found, claimed, reason = ask_safety(chunk, f'[구간 {index}/4]\n', instruction_first)
        merged |= found
        claimed_total = None if claimed is None or claimed_total is None else claimed_total + claimed
        reasons.append(reason)
    return merged, claimed_total, sorted(set(reasons))


for number in [1, 2, 3]:
    print(f'[나눠 넣기 {number}번째]', compare(*ask_chunks()))

[나눠 넣기 1번째] 표의 줄 14개(모델이 적은 개수 14), 정답 14/14, 빠진 줄 0개, 정답이 아닌 줄 0개(그중 "재시작 후 정상" 기록 0개), 종료 이유 ['end']
[나눠 넣기 2번째] 표의 줄 14개(모델이 적은 개수 14), 정답 14/14, 빠진 줄 0개, 정답이 아닌 줄 0개(그중 "재시작 후 정상" 기록 0개), 종료 이유 ['end']
[나눠 넣기 3번째] 표의 줄 14개(모델이 적은 개수 14), 정답 14/14, 빠진 줄 0개, 정답이 아닌 줄 0개(그중 "재시작 후 정상" 기록 0개), 종료 이유 ['end']


지시를 일지 앞에 두면 결과가 달라지는지 봅니다. 위의 나눠 넣기와 구간 나누기는 같고, 지시와 일지의 순서만 바꿉니다.

In [ ]:
# 예제 2-6 (지시를 일지 앞에 두고 나눠 넣습니다)
for number in [1, 2, 3]:
    print(f'[지시를 앞에 두고 나눠 넣기 {number}번째]', compare(*ask_chunks(instruction_first=True)))

[지시를 앞에 두고 나눠 넣기 1번째] 표의 줄 26개(모델이 적은 개수 26), 정답 14/14, 빠진 줄 0개, 정답이 아닌 줄 12개(그중 "재시작 후 정상" 기록 12개), 종료 이유 ['end']
[지시를 앞에 두고 나눠 넣기 2번째] 표의 줄 14개(모델이 적은 개수 14), 정답 14/14, 빠진 줄 0개, 정답이 아닌 줄 0개(그중 "재시작 후 정상" 기록 0개), 종료 이유 ['end']
[지시를 앞에 두고 나눠 넣기 3번째] 표의 줄 22개(모델이 적은 개수 22), 정답 14/14, 빠진 줄 0개, 정답이 아닌 줄 8개(그중 "재시작 후 정상" 기록 8개), 종료 이유 ['end']


400줄(약 1만 4천 자)은 컨텍스트 윈도우 안에 들어갔고 응답도 잘리지 않았습니다(종료 이유 `end`). 한 번에 넣은 세 번은 모두 정답 14건을 빠짐없이 찾았고 정답이 아닌 줄이 없었습니다. 모델이 적은 "찾은 개수"(14)도 표의 줄 수와 같았습니다. 100줄씩 네 구간으로 나눠 넣고 합친 세 번도 같았습니다. 지시를 일지 앞에 두고 나눠 넣은 세 번은 달랐습니다. 세 번 모두 정답 14건은 빠짐없이 찾았지만, 두 번은 정답이 아닌 줄이 12개와 8개 섞였고 모두 "재시작 후 정상" 기록이었습니다. 이때도 모델이 적은 "찾은 개수"(26, 22)는 표의 줄 수와 같았습니다. 지시를 뒤에 두고 나눠 넣은 조건에서는 이 줄이 섞이지 않았지만, 지시를 앞에 두고 한 번에 넣는 조건은 실행하지 않았으므로 원인이 지시의 위치인지 나눈 것인지는 구분할 수 없습니다. 공식 문서가 긴 문서를 앞에, 지시를 뒤에 두라고 안내하는 것과 같은 방향이지만, 세 번 가운데 두 번에서 나타난 결과이므로 일반화하지 않습니다. 한 번에 넣은 결과가 맞았으므로 이 일지는 나눌 이유가 없는 경우입니다.

나누는 방법은 입력이 컨텍스트 윈도우에 들어가지 않거나 응답이 잘릴 때 씁니다. 나눈 결과가 한 번에 넣은 결과와 같다는 보장은 없습니다. 모델이 구간마다 따로 판단하므로, 애매한 줄을 넣을지 뺄지의 기준을 구간 지시에 적고 합친 결과를 정답이나 원문과 대조합니다. 이 일지에서는 "알람 1회, 재시작 후 정상" 기록 26줄이 그런 줄입니다. 이번 실행에서는 그 기준을 적지 않은 구간 지시로 이 기록이 안전 항목에 들어갔습니다. 2026-11-23 실행에서는 같은 코드로 세 조건 모두 이 기록이 들어간 일이 없었으므로, 같은 프롬프트라도 이 기록이 들어가는지는 실행마다 다릅니다.

이번에는 정답 파일이 있어서 정답이 아닌 줄을 셀 수 있었습니다. 실무에서는 정답이 없으므로 구간별 개수의 합과 합친 목록의 항목 수가 같은지로는 틀린 줄을 찾지 못합니다. 합친 목록에서 의심스러운 유형(여기서는 "재시작 후 정상" 기록)을 원문에서 찾아 몇 줄을 직접 대조합니다.

## 프롬프트 체인

> **요점**
>
> 한 작업을 단계로 나눠 앞 단계의 결과를 다음 단계에 넘기면 단계마다 결과를 확인할 수 있습니다. 한 단계에는 한 가지만 시키고, 중간 산출물을 남기고, 끝에서 항목 수를 세는 일은 코드로 합니다.

프롬프트 체인은 한 작업을 여러 단계로 나누고, 앞 단계의 출력을 다음 단계의 입력으로 넘기는 방법입니다. 회의록 정리나 주간 보고처럼 절차가 정해진 일에 씁니다. 한 번에 "정리해줘"라고 요청하면 결과가 틀렸을 때 어느 부분에서 빠지고 틀렸는지 찾을 수 없습니다. 단계로 나누면 단계마다 결과를 확인하고, 틀린 단계만 다시 실행할 수 있습니다. Anthropic의 공식 문서도 중간 산출물을 확인하거나 단계별로 기록하고 분기해야 할 때 체인이 유용하다고 설명합니다(확인일 2026-11-23).

이 파트의 체인은 회의 받아쓰기 26줄을 네 단계로 정리합니다. 단계는 차례로 1단, 2단, 3단, 4단으로 줄여 부릅니다. 네 단계는 같은 대화에서 보냅니다. 앞 단계의 응답이 대화 이력으로 다음 요청에 함께 전송되므로, 앞 단계의 출력이 다음 단계의 입력이 됩니다. 같은 대화에서 3단을 다시 보내면 1단, 2단과 함께 처음 보낸 3단의 질문과 응답도 이력에 남은 채로 전송됩니다. 3단만 새로 하려면 `chat.history`에서 3단의 두 메시지(질문과 응답)를 지우고 보내거나, 2단까지의 응답을 붙여 넣은 새 요청을 보냅니다. Anthropic의 공식 문서는 체인의 각 단계를 별도의 API 요청으로 설명합니다. 단계를 서로 독립된 요청으로 보내려면 앞 단계의 응답을 다음 프롬프트에 직접 붙여 넣습니다.



*체인은 네 단계로 이루어집니다. 청록색 상자인 4단에서 처음 항목 수와 표의 행 수를 비교합니다. 모델이 답한 개수는 코드로 센 개수와 비교하고, 둘이 다르면 코드로 센 개수를 믿습니다.*

먼저 비교용으로 체인 없이 한 번에 정리하게 합니다. 응답에 받아쓰기에 없는 요일 표기(`8/14(목)` 같은 것)가 덧붙었는지 코드로 찾습니다. 그다음 네 단계를 같은 대화에서 차례로 보냅니다. 1단의 항목 수와 2단이 만든 표에서 센 행 수는 모델에게 묻지 않고 코드로 세고, 4단에서 모델이 답한 개수와 나란히 놓습니다.

**예제 2-7. 받아쓰기를 한 번에 정리하기와 네 단계 체인으로 정리하기**

In [ ]:
# 예제 2-7
import re

dictation = read_material('회의_받아쓰기_8월2주차.txt')
print('\n'.join(dictation.splitlines()[:8]))
print('...')

# 체인 없이 한 번에 정리를 요청합니다
chat = llm.Chat()
chat.say('다음 회의록 정리해줘.\n\n' + dictation)
invented = re.findall(r'\d+/\d+\s*\([월화수목금토일]\)', chat.last.text)
print(f'\n받아쓰기에 없는 요일 표기: {invented or "없음"}')

[8월 2주차 장비팀 주간회의 · 받아쓰기]
(최파트장) 자 시작합시다. 지난주 3호기 인터락 건부터.
(이OO) 보전팀 점검 결과 센서 접점 불량이었고요, 부품은 8/14 입고됩니다.
(최파트장) 교체는 언제 하죠?
(이OO) 8/15 PM 때 같이 하려고 합니다. 라인 세우는 김에.
(최파트장) 좋습니다. 그럼 8/15 이후로 인터락 재발하면 바로 알려주세요.
(박OO) 저 잠깐, 5호기 E-201 얘기 좀. 이번 주만 네 번 떴습니다.
(최파트장) 벤더 회신은요?
...
# 8월 2주차 장비팀 주간회의록

**참석자:** 최파트장, 이OO, 박OO, 정OO, 김OO

---

## 1. 논의 내용

| 안건 | 현황 | 결정/방향 |
|---|---|---|
| **3호기 인터락** | 보전팀 점검 결과 센서 접점 불량. 교체 부품은 8/14 입고 예정 | 8/15 PM 때 교체. 이후 재발하면 즉시 공유 |
| **5호기 E-201 알람** | 이번 주에만 4회 발생. 벤더 회신: 펌프 압력 센서 캘리브레이션 필요. 캘리 장비가 현재 3호기에 사용 중 | 장비 사용 순번 조정 필요. 보전팀과 협의 |
| **신규 레시피 A-7** | 3랏 진행. 두께 산포가 기존보다 커 보이나 판단하기에는 이름 | 3랏 추가 진행 후 판단. 다음주 화요일 리뷰에서 재논의 |
| **방진복 재고** | 12벌 남음. 다음주 소진 예상 | 20벌 발주 완료 |
| **폐액 라벨 (환경팀 요청)** | 환경팀에서 재연락. 금주 내 회신 필요. 담당이 장비팀인지 공정팀인지 불분명 | 담당 부서 확인 후 대응 |
| **다음주 야간조 인력** | 휴가로 1명 부족 | C조에서 1명 지원받는 방향으로 협의 |

---

## 2. Action Items

| 담당 | 내용 | 기한 |
|---|---|---|
| 이OO | 폐액 라벨 건 담당 부서(장비팀/공정팀) 확인 후 공유 | **내일** |
| 정OO | 캘리 장비 순번 조정을 위해 보전팀과 협의 (5호기 E-201) | 

1단입니다. 결정과 미결로 나누고, 항목을 번호 붙여 적게 합니다. 항목 수는 코드로 셉니다.

In [ ]:
# 예제 2-7 (1단: 결정과 미결로 나눕니다)
chat = llm.Chat()
chat.say('다음 회의 받아쓰기를 두 가지로 분류해줘.\n\n[결정] 이미 정해진 것\n[미결] 아직 안 정해진 것\n\n'
         '항목은 각각 1번부터 "1. 원문 표현" 형식으로 한 줄에 하나씩 적고, 원문 표현을 유지해.\n\n' + dictation)
decided, _, undecided = chat.last.text.partition('[미결]')
count_decided = len(re.findall(r'^\s*\d+\.', decided, flags=re.M))
count_undecided = len(re.findall(r'^\s*\d+\.', undecided, flags=re.M))
print(f'\n코드로 센 개수: 결정 {count_decided}건, 미결 {count_undecided}건')

**[결정]**
1. 부품은 8/14 입고됩니다.
2. 8/15 PM 때 같이 하려고 합니다.
3. 8/15 이후로 인터락 재발하면 바로 알려주세요.
4. 정OO님이 보전팀이랑 협의해 주시겠어요? 네 이번 주 내로 하겠습니다.
5. 3랏 더 하고 보는 게 맞을 것 같습니다.
6. 다음주 화요일 리뷰 때 다시 봅시다.
7. 20벌 발주 넣었습니다. 완료됐습니다.
8. 이OO님이 확인해서 내일까지 알려주세요.
9. 그건 제가 하겠습니다.

**[미결]**
1. 순번 조정이 필요하겠네요.
2. 두께 산포가 기존보다 커 보입니다.
3. 금주 내 회신해야 합니다.
4. 저희 쪽인지 공정팀인지 불분명합니다. 확인이 필요합니다.
5. C조에서 한 명 빌려오면 될 것 같은데 협의가 필요합니다.

코드로 센 개수: 결정 9건, 미결 5건


2단입니다. 결정 항목에서 액션아이템 표를 만들게 하고, 표에서 센 행 수와 빠진 항목을 코드로 확인합니다. 받아쓰기에서 담당이 말로 정해진 액션아이템은 네 건이고, 방진복 발주는 이미 끝난 일이라 표에 있으면 안 됩니다.

In [ ]:
# 예제 2-7 (2단: 액션아이템 표를 만들게 하고 코드로 확인합니다)
chat.say('위 [결정] 항목에서 액션아이템만 뽑아 마크다운 표로 정리해줘.\n열: 담당 | 할 일 | 기한\n'
         "회의록에 기한이 없으면 '미정'으로. 담당이 불명확하면 '확인 필요'로.")
table_rows = [line for line in chat.last.text.splitlines()
              if line.strip().startswith('|') and not re.match(r'^\|[\s:|-]+$', line.strip())]
table_count = len(table_rows) - 1                              # 첫 줄은 머리글입니다
print(f'\n코드로 센 표의 행 수: {table_count}')

# 받아쓰기에서 담당이 말로 정해진 액션아이템 네 건의 담당과, 표의 담당을 비교합니다
cells = [[cell.strip() for cell in row.strip().strip('|').split('|')] for row in table_rows[1:]]
cells = [row for row in cells if len(row) >= 2]
expected = {'인터락': '이OO', 'E-201': '정OO', '폐액': '이OO', '야간조': '최파트장'}
for keyword, owner in expected.items():
    matched = [row[0] for row in cells if keyword in row[1]]
    print(f'{keyword} 건: 표의 담당 {matched or "행 없음"}, 받아쓰기의 담당 {owner}')
print('담당 칸에 적힌 값 전체:', sorted({row[0] for row in cells}))
print('이미 끝난 일(방진복 발주)이 표에 있음:', any('방진복' in row[1] for row in cells))

| 담당 | 할 일 | 기한 |
|---|---|---|
| 이OO | 3호기 인터락 센서 부품 교체 (8/15 PM 때 같이 진행) | 8/15 |
| 확인 필요 | 8/15 이후 3호기 인터락 재발 시 바로 공유 | 8/15 이후 재발 시 즉시 |
| 정OO | 5호기 E-201 캘리 장비 순번 조정을 위해 보전팀과 협의 | 이번 주 내 |
| 박OO | 신규 레시피 A-7 3랏 추가 진행 | 미정 |
| 확인 필요 | 레시피 A-7 결과 리뷰 (두께 산포 재확인) | 다음주 화요일 |
| 이OO | 폐액 라벨 건 담당 부서(장비팀/공정팀) 확인 후 공유 | 내일 |
| 최파트장 | 다음주 야간조 인력 부족 관련해 C조와 협의 | 미정 |

**제외한 항목**
- "부품은 8/14 입고됩니다": 예정 사실 공유라서 제외했습니다.
- "20벌 발주 넣었습니다. 완료됐습니다.": 이미 완료된 일이라 제외했습니다.

**참고**
- "8/15 이후 인터락 재발 시 공유"는 최파트장이 대상을 특정하지 않고 요청해서 담당을 '확인 필요'로 두었습니다.
- A-7 리뷰는 "다음주 화요일 리뷰 때 다시 봅시다"라고만 해서 담당이 특정되지 않았습니다.

코드로 센 표의 행 수: 7
인터락 건: 표의 담당 ['이OO', '확인 필요'], 받아쓰기의 담당 이OO
E-201 건: 표의 담당 ['정OO'], 받아쓰기의 담당 정OO
폐액 건: 표의 담당 ['이OO'], 받아쓰기의 담당 이OO
야간조 건: 표의 담당 ['최파트장'], 받아쓰기의 담당 최파트장
담당 칸에 적힌 값 전체: ['박OO', '이OO', '정OO', '최파트장', '확인 필요']
이미 끝난 일(방진복 발주)이 표에 있음: False


3단입니다. 표에서 담당이나 기한이 비어 있는 행만 다시 보여 주고, 누구에게 무엇을 물을지 적게 합니다.

In [ ]:
# 예제 2-7 (3단: 빠진 정보를 되묻게 합니다)
chat.say("위 표에서 '확인 필요' 또는 '미정'이 들어간 행만 다시 보여주고, "
         '각각 누구에게 무엇을 물어야 하는지 한 줄로 적어줘.')

| 담당 | 할 일 | 기한 |
|---|---|---|
| 확인 필요 | 8/15 이후 3호기 인터락 재발 시 바로 공유 | 8/15 이후 재발 시 즉시 |
| 박OO | 신규 레시피 A-7 3랏 추가 진행 | 미정 |
| 확인 필요 | 레시피 A-7 결과 리뷰 (두께 산포 재확인) | 다음주 화요일 |
| 최파트장 | 다음주 야간조 인력 부족 관련해 C조와 협의 | 미정 |

**확인할 사항**
1. 인터락 재발 공유: 최파트장에게, 재발 시 누가 누구에게 알려야 하는지 물어보세요.
2. A-7 3랏 추가 진행: 박OO님에게, 3랏을 언제까지 진행하고 결과를 낼 수 있는지 물어보세요.
3. A-7 결과 리뷰: 최파트장에게, 화요일 리뷰에서 A-7 결과를 정리해 발표할 담당자가 누구인지 물어보세요.
4. C조 협의: 최파트장에게, 다음주 야간조 일정 전 언제까지 협의를 마칠지 물어보세요.


4단입니다. 모델에게 개수를 비교하게 하고, 코드로 센 값을 나란히 놓습니다.

In [ ]:
# 예제 2-7 (4단: 개수를 셉니다)
chat.say('처음 [결정]·[미결]로 나눈 항목 수와 액션아이템 표의 행 수를 비교해줘. '
         '표에 들어가지 않은 [결정] 항목이 있으면 무엇인지, 왜 빠졌는지 알려줘. '
         "마지막으로 표에서 담당이 '확인 필요'인 행이 몇 개인지도 적어줘.")
unknown_owner = len([row for row in cells if row[0] == '확인 필요'])
print(f'\n코드로 센 값: [결정] {count_decided}건, [미결] {count_undecided}건, 표 {table_count}행, '
      f'담당이 확인 필요인 행 {unknown_owner}개')

**항목 수 비교**

| 구분 | 개수 |
|---|---|
| [결정] | 9개 |
| [미결] | 5개 |
| 액션아이템 표 행 수 | 7행 |

표는 [결정] 항목에서만 뽑았기 때문에 [미결] 5개는 표에 들어가지 않았습니다.

**표에 들어가지 않은 [결정] 항목 (2개)**
- 결정 1번 "부품은 8/14 입고됩니다.": 입고 예정을 알리는 사실 공유라서 누가 할 일이 아니어서 뺐습니다.
- 결정 7번 "20벌 발주 넣었습니다. 완료됐습니다.": 이미 끝난 일이라 남은 할 일이 없어서 뺐습니다.

따라서 [결정] 9개 중 7개가 표의 7행이 되었고, 항목과 행이 1:1로 대응합니다.

**담당이 '확인 필요'인 행: 2개**
1. 8/15 이후 3호기 인터락 재발 시 바로 공유
2. 레시피 A-7 결과 리뷰 (두께 산포 재확인)

코드로 센 값: [결정] 9건, [미결] 5건, 표 7행, 담당이 확인 필요인 행 2개


체인 없이 한 번에 정리한 응답은 회의록 형식의 문서였고, 끝의 Action Items 표는 6행이었습니다. 받아쓰기에 없는 요일 표기는 없었습니다. 이번 실행에서는 한 번에 정리한 결과도 담당이 말로 정해진 네 건(인터락 부품 교체, E-201 협의, 폐액 라벨 확인, 야간조 조정)을 모두 담았습니다. 다만 재발 시 공유의 담당은 `전원`, A-7 추가 진행의 담당은 `박OO`로 적었습니다. 체인이 더 많이 찾아낸 것은 아닙니다. 체인의 이점은 단계마다 결과가 따로 남아 코드가 검사할 수 있다는 점입니다.

1단에서 결정 9건과 미결 5건이 나왔습니다. 항목은 문장 단위로 나뉘었고, 결정 4번처럼 질문과 답을 한 항목에 합친 경우도 있어 "한 줄에 하나" 지시와 어긋납니다. E-201 순번 조정(결정 4번, 미결 1번), A-7(결정 5번과 6번, 미결 2번), 폐액 라벨(결정 8번, 미결 3번과 4번), 야간조 인력(결정 9번, 미결 5번)이 결정과 미결 양쪽에 들어갔습니다. 담당과 기한이 정해진 부분은 결정에, 풀리지 않은 부분은 미결에 넣었기 때문입니다.

2단의 표는 7행이었고, 담당이 말로 정해진 네 건이 모두 있었으며 이미 끝난 방진복 발주는 없었습니다. 코드로 담당 칸을 비교하면 E-201 건은 정OO, 폐액 건은 이OO, 야간조 건은 최파트장으로 받아쓰기와 같았습니다. 인터락 건에는 이OO의 행과 "확인 필요"의 행이 하나씩 있는데, 후자는 재발 시 공유라는 다른 일의 행입니다. 담당 칸에 적힌 값 전체를 출력해 보니 박OO도 있었습니다. 받아쓰기에서 박OO는 "3랏"(3로트)을 진행했다고 말하고 3랏을 더 해 보자고 제안한 사람이고, 추가 진행의 담당으로 정해진 적은 없습니다. 이번 실행에서는 한 번에 정리한 응답도 같은 일의 담당을 박OO로 적었습니다(2026-11-23 실행에서는 "담당 미지정"으로 적었습니다). 키워드가 표에 있는지만 확인하는 코드로는 박OO가 담당으로 잘못 적힌 것을 찾아내지 못합니다.

4단에서 모델이 답한 결정 9건, 미결 5건, 표 7행, 담당이 "확인 필요"인 행 2개는 코드로 센 값과 같았습니다. 표에 들어가지 않은 결정 두 건은 부품 입고 예정(사실 공유)과 방진복 발주(이미 완료)라고 밝혔습니다. 2026-11-23 실행에서는 2단 응답 끝의 설명이 담당이 "확인 필요"인 행 하나를 빠뜨렸고, 3단 응답 끝에서 모델이 이를 정정한 일이 있었습니다. 이번에 모델이 센 값은 맞았지만, 같은 체인에서도 모델이 개수를 잘못 적는 일이 있으므로 개수는 코드로 확인합니다. 모델이 센 값과 코드로 센 값이 어긋나면 맞는 쪽은 코드로 센 값입니다.

## 검증: 자기 검증과 루브릭

> **요점**
>
> 모델이 자기 응답을 사실, 확인 필요, 추측으로 나누게 하면 어디부터 확인할지 표시되지만 확인을 대신하지는 않습니다. 다른 사람에게 전달하는 문서는 숫자와 고유명사를 근거 자료(원문이나 공식 문서)와 대조하고, 프롬프트를 고친 효과는 루브릭으로 여러 번 채점해 판단합니다.

모델의 응답은 맞을 때와 틀릴 때 말투가 같으므로 검증이 필요합니다. 검증은 세 단계로 나눕니다.

<img alt="검증의 세 단계. 1단계는 모델이 스스로 응답을 세 가지로 분류하며 가장 빠르지만 놓치는 문장이 있다. 2단계는 사람이 무엇을 확인할지 고른다. 3단계는 고른 숫자와 고유명사를 근거 자료와 대조하며 가장 확실하고 다른 사람에게 전달하는 문서에만 한다" width="700" src="">

*오른쪽 단계일수록 확실하지만 비용이 듭니다. 1단계는 어디부터 확인할지 표시할 뿐이고, 다른 사람에게 전달하는 문서는 3단계까지 거칩니다.*

### 자기 검증

자기 검증은 모델에게 자기 응답을 사실, 확인 필요, 추측으로 나누게 하는 단계입니다. 응답의 문장이 모두 같은 확신으로 쓰여 있으므로, 확인해야 할 문장이 어디인지 모델이 표시해 주면 사람이 어디부터 볼지 정할 수 있습니다.

그러나 모델은 틀린 내용을 틀린 줄 모르고 쓸 수 있습니다. 모델이 "널리 알려진 사실"로 분류한 문장에도 틀린 내용이 있을 수 있고, 모델이 덧붙인 숫자는 모델이 지어냈을 수 있습니다. 그래서 분류 결과를 믿지 않고 분류 결과에 나온 숫자와 고유명사를 코드와 사람이 원문과 대조합니다.

**예제 2-8. 응답을 사실, 확인 필요, 추측으로 나누게 하기**

In [ ]:
# 예제 2-8
import re

claim = '반도체 공정에서 파티클 기준은 0.1마이크로미터 이상 입자를 웨이퍼당 5개 이하로 관리한다'
chat = llm.Chat()
chat.say(f'다음 문장이 사실인지 확인하고, 확실하지 않은 부분을 표시해줘.\n\n"{claim}"')

## 판정: 일반적인 사실로 보기 어렵습니다

이 문장은 특정 조건에서는 맞을 수 있지만, 업계 공통 기준처럼 쓰면 부정확합니다.

### 부분별 검토

| 표현 | 판정 | 이유 |
|---|---|---|
| "파티클 기준을 관리한다" | ✅ 사실 | 파티클 관리는 반도체 공정의 핵심 항목입니다. |
| "0.1 µm 이상" | ⚠️ 불확실 | 관리 크기는 노드, 공정 단계, 장비에 따라 다릅니다. 첨단 공정은 수십 nm 수준까지 내려가고, 오래된 공정은 더 큰 크기를 쓰기도 합니다. |
| "웨이퍼당 5개 이하" | ⚠️ 불확실 | 개수 기준도 공정 단계, 장비, 웨이퍼 크기, 팹마다 다릅니다. 보편적으로 통용되는 단일 수치는 확인되지 않습니다. |
| 문장 전체의 일반화 | ❌ 부정확 | 하나의 수치가 모든 반도체 공정에 적용되는 것처럼 읽힙니다. |

### 확실하지 않은 이유

1. **기준이 하나가 아닙니다.**
   - 클린룸 공기 청정도는 ISO 14644-1 기준으로 **m³당 입자 수**로 규정합니다. 예를 들어 ISO Class 3은 0.1 µm 이상 입자를 m³당 1,000개까지 허용합니다.
   - 장비 파티클 성능은 보통 **웨이퍼 1장 통과 시 추가되는 입자 수(adder)**로 봅니다.
   - 입고 웨이퍼 품질은 **LPD(Light Point Defect)** 기준으로 봅니다.
   - 단위가 m³, 웨이퍼당, cm²당으로 서로 다르므로 직접 비교할 수 없습니다.

2. **측정 방식에 따라 값이 달라집니다.** 검사 장비의 감도와 "LSE(light scattering equivalent) 크기" 환산 방식에 따라 같은 웨이퍼도 결과가 다릅니다. 막질과 표면 상태에도 영향을 받습니다.

3. **공정과 팹에 따라 다릅니다.** 식각, CVD, CMP, 세정, 포토 등 단계마다 기준이 다르고, 같은 단계라도 회사와 제품마다 다릅니다. 대부분 사내 사양이라 공개되지 않습니다.

4. **"관리 기준"의 의미가 모호합니다.** 스

응답을 세 가지로 나누게 합니다.

In [ ]:
# 예제 2-8 (응답을 세 가지로 분류하게 합니다)
chat.say('위 답변에서\n1. 널리 알려진 사실\n2. 문서로 확인이 필요한 것\n3. 네가 추측한 것\n을 나눠서 표시해줘.')

먼저 전제부터 밝히면, 이전 답변은 외부 자료를 검색하거나 확인하지 않고 제가 학습한 지식만으로 작성했습니다. 그 점을 감안해 나눴습니다.

## 1. 널리 알려진 사실

- 파티클 관리는 반도체 공정의 핵심 관리 항목입니다.
- 관리해야 하는 입자 크기는 노드, 공정 단계, 장비에 따라 달라집니다.
- 클린룸 청정도는 ISO 14644-1에서 **단위 부피(m³)당 입자 수**로 규정합니다.
- 장비 파티클 성능을 웨이퍼에 추가되는 입자 수(adder)로 평가하는 것은 업계 관행입니다.
- LPD(Light Point Defect)는 웨이퍼 표면 검사에서 쓰는 용어입니다.
- 광산란 검사에서는 LSE(PSL 등가 크기) 개념을 쓰며, 검사 장비 감도와 표면 상태(막질, haze 등)에 따라 측정값이 달라집니다.
- 단위가 m³, 웨이퍼당, cm²당으로 서로 달라 직접 비교할 수 없다는 것은 논리적으로 당연합니다.

## 2. 문서로 확인이 필요한 것

| 항목 | 확인할 문서 |
|---|---|
| ISO Class 3의 0.1 µm 허용 농도 1,000개/m³ | ISO 14644-1 (2015판) 표. 기억에 근거한 값이라 원문 대조가 필요합니다. |
| 입고 웨이퍼 LPD 기준의 구체적 크기, 단위(웨이퍼당 또는 cm²당) | SEMI 웨이퍼 사양, 웨이퍼 공급사 사양서 |
| 첨단 공정의 관리 크기가 "수십 nm 수준" | 장비사 자료, 학회 자료 등. 대략적 경향일 뿐 구체적 수치는 아닙니다. |
| 장비 인증 시 adder 기준의 실제 크기와 개수 | 장비사 사양서, 팹 사내 규격 |
| 원문 "0.1 µm 이상, 웨이퍼당 5개 이하"의 출처와 적용 조건 | 원문 작성자 또는 해당 문서 |

## 3. 제가 추측하거나 판단한 것

- **"대부분 사내 사양이라 공개되지 않는다"**: 근거 자료 없이 업계 관행을 짐작한 것입니다.
- **"식각, CVD, CMP, 세정, 포토 단계마다 기준이 다르다"**: 일반적인 공정 지식에서 추론한 것으로, 단

분류한 응답에서 질문에 없던 숫자를 코드로 찾습니다. 모델이 덧붙인 숫자는 확인 대상입니다.

In [ ]:
# 예제 2-8 (질문에 없던 숫자를 찾습니다)
def numbers_in(text):
    return set(re.findall(r'\d+(?:\.\d+)?', text))


print('분류한 응답에 나온 숫자 중 질문에 없던 것:', sorted(numbers_in(chat.last.text) - numbers_in(claim)))

분류한 응답에 나온 숫자 중 질문에 없던 것: ['000', '1', '14644', '2', '2015', '3']


첫 응답은 이 문장을 업계 공통 기준처럼 쓰면 부정확하다고 판정했고, 0.1마이크로미터와 웨이퍼당 5개를 "불확실"로 표시했습니다. 분류를 요청하자 응답은 먼저 이전 답변이 외부 자료를 확인하지 않고 학습한 지식만으로 작성한 것이라고 밝히고, 첫 응답의 "부정확" 판정이 과하게 단정한 표현이었다고 고쳤습니다. 정확하게는 "일반 기준으로 뒷받침할 근거를 찾지 못했다"는 수준이라는 것입니다. 모델이 입장을 바꾼 것이지만, 이 응답만으로는 어느 쪽이 맞는지 알 수 없습니다. 분류 결과도 확인할 문장이 줄어든 것은 아닙니다. 사실 7개, 확인 필요 5개, 추측 5개로 나뉘어 확인 항목이 오히려 늘었습니다. 자기 검증이 한 일은 문장마다 근거의 수준을 표시한 것이고, 그 표시가 맞는지는 다시 확인해야 합니다.

"널리 알려진 사실"로 분류된 항목에도 확인할 주장이 들어 있습니다. 예를 들어 ISO 14644-1이 단위 부피(m³)당 입자 수로 청정도를 규정한다는 항목이 그렇습니다. 코드로 질문에 없던 숫자를 찾으니 `1`, `2`, `3`, `000`, `14644`, `2015`가 나왔습니다. `1`, `2`, `3`은 목록 번호이고, `1`은 "ISO 14644-1"과 "1,000개"에도 있습니다. `000`은 "1,000"이 쉼표에서 나뉜 것으로, 숫자를 찾는 정규식이 천 단위 쉼표를 처리하지 않기 때문입니다. 목록 번호를 빼고 남는 숫자(`14644`, `2015`, ISO Class 3의 허용 농도 1,000개)는 모델이 덧붙인 주장이므로 확인할 대상입니다. 모델도 1,000개는 기억에 근거한 값이라며 "문서로 확인이 필요한 것"에 넣었습니다. 이 질문에는 대조할 원문이 없으므로, 대조할 자료는 표준 문서나 장비 사양서입니다. 분류 결과를 믿고 확인을 건너뛰지 않고, 다른 사람에게 전달하는 문서에서는 숫자와 고유명사를 근거 자료와 대조합니다.

### 루브릭으로 고치기 전후를 비교하기

프롬프트를 고친 뒤 "좋아진 것 같다"는 느낌은 사람마다 다릅니다. 루브릭은 결과를 몇 개 항목으로 나누고 항목마다 점수의 기준을 적은 평가표입니다. 같은 루브릭으로 고치기 전후를 채점하면 비교의 기준이 같아집니다.

루브릭의 세 항목은 빠짐없음, 정확함, 바로 쓸 수 있음이고, 항목마다 1~3점입니다. 채점도 모델에게 맡길 수 있습니다. 이 방법을 LLM-as-a-judge라고 합니다. 모델의 채점도 응답이므로 실행마다 달라지고, 요약이 놓인 순서가 점수에 영향을 줄 수도 있습니다(position bias). 같은 채점을 여러 번 하고 점수의 분포를 보며 판단합니다. 예제 2-1의 0단계 응답과 4단계 응답을 채점합니다.

**예제 2-9. 루브릭으로 채점하고 프롬프트 템플릿으로 정리하기**

In [ ]:
# 예제 2-9
import re


def grading_prompt(criteria, first='0단계 지시만', second='4단계 + 제약', with_reason=False):
    """요약 두 개를 같은 루브릭으로 채점하게 하는 프롬프트를 만듭니다. first를 요약 A, second를 요약 B로 보여 줍니다."""
    rubric = '\n'.join(f'- {name}: {rule}' for name, rule in criteria.items())
    shape = '{' + ', '.join(f'"{name}": 점수' for name in criteria)
    shape += ', "근거": "점수가 가장 낮은 항목의 근거를 요약에서 한 구절 인용"}' if with_reason else '}'
    return ('아래 [원본 회의록]과 요약 두 개를 [채점표]로 채점해줘.\n\n[채점표] 항목마다 1~3점\n' + rubric + '\n\n'
            f'출력은 다른 말 없이 JSON 한 개로만 적어줘. 형식: {{"A": {shape}, "B": {shape}}}\n'
            '점수를 줄 때 요약 A와 B 가운데 어느 쪽이 나중에 쓴 것인지는 고려하지 마.\n\n'
            f'[원본 회의록]\n{minutes}\n\n[요약 A]\n{summaries[first]}\n\n[요약 B]\n{summaries[second]}')


def grade_many(criteria, runs=5, first='0단계 지시만', second='4단계 + 제약'):
    """요약 두 개를 같은 루브릭으로 runs번 채점합니다."""
    prompt = grading_prompt(criteria, first, second)
    pairs = (('A', first), ('B', second))
    totals, items = [], []
    for _ in range(runs):
        chat = llm.Chat(model='소형')
        chat.say(prompt, show=False)
        scores = json.loads(re.search(r'\{.*\}', chat.last.text, flags=re.S).group())
        totals.append({stage: sum(scores[label].values()) for label, stage in pairs})
        items.append({(stage, name): scores[label][name] for label, stage in pairs for name in criteria})
    index = [f'{number}회' for number in range(1, runs + 1)]
    item_scores = pd.DataFrame(items, index=index)
    item_scores.columns = pd.MultiIndex.from_tuples(item_scores.columns)     # 열을 (요약, 항목) 두 단계로 두어야 unstack으로 요약별 항목 평균을 펼칠 수 있습니다
    return pd.DataFrame(totals, index=index), item_scores.mean().unstack(0).reindex(list(criteria))


def summarize(result):
    totals, items = result
    first, second = totals.columns
    print(totals.T)
    print(f'{first}의 합계가 높음 {(totals[first] > totals[second]).sum()}번, 같음 {(totals[first] == totals[second]).sum()}번, '
          f'{second}의 합계가 높음 {(totals[first] < totals[second]).sum()}번')
    print('항목별 평균 점수')
    print(items.round(1), '\n')


minutes = read_material('회의록_8월1주차.txt')
criteria = {'빠짐없음': '1점 원자료 항목이 여럿 누락 / 2점 일부 누락 / 3점 전부 반영',
            '정확함': '1점 없는 내용이 생김 / 2점 애매한 표현 / 3점 원자료와 일치',
            '바로 쓸 수 있음': '1점 다시 써야 함 / 2점 조금 고치면 됨 / 3점 그대로 붙임'}
summarize(grade_many(criteria))

          1회  2회  3회  4회  5회
0단계 지시만    7   7   7   7   7
4단계 + 제약   6   7   7   7   6
0단계 지시만의 합계가 높음 2번, 같음 3번, 4단계 + 제약의 합계가 높음 0번
항목별 평균 점수
           0단계 지시만  4단계 + 제약
빠짐없음           3.0       3.0
정확함            2.0       2.0
바로 쓸 수 있음      2.0       1.6 


두 요약이 놓이는 순서가 점수에 영향을 주는지 보려고, 요약 A와 B를 바꿔 같은 채점을 합니다.

In [ ]:
# 예제 2-9 (요약의 순서를 바꿔 채점합니다)
summarize(grade_many(criteria, first='4단계 + 제약', second='0단계 지시만'))

          1회  2회  3회  4회  5회
4단계 + 제약   7   7   7   7   7
0단계 지시만    7   8   7   7   9
4단계 + 제약의 합계가 높음 0번, 같음 3번, 0단계 지시만의 합계가 높음 2번
항목별 평균 점수
           0단계 지시만  4단계 + 제약
빠짐없음           3.0       3.0
정확함            2.2       2.0
바로 쓸 수 있음      2.4       2.0 


점수만 받으면 점수가 낮은 이유를 알 수 없습니다. 점수와 함께 요약에서 근거를 한 구절 인용하게 해 한 번 보냅니다.

In [ ]:
# 예제 2-9 (점수와 함께 근거를 인용하게 합니다)
chat = llm.Chat(model='소형')
chat.say(grading_prompt(criteria, with_reason=True))

{"A": {"빠짐없음": 3, "정확함": 2, "바로 쓸 수 있음": 3, "근거": "참고로 2라인 파티클 상승은 원인이 불명확해서, 필터 교체 후에도 수치를 계속 확인해야 할 것 같습니다."}, "B": {"빠짐없음": 3, "정확함": 2, "바로 쓸 수 있음": 2, "근거": "가동률 관련 리스크는 3건입니다."}}


팀장이 가동률 리스크를 가장 궁금해한다는 목적을 항목으로 더해 다시 채점합니다.

In [ ]:
# 예제 2-9 (목적을 항목으로 더해 채점합니다)
with_purpose = {**criteria, '목적 맞춤': '1점 가동률 리스크 안건이 묻힘 / 2점 일부만 앞에 옴 / 3점 맨 앞에 모여 있음'}
summarize(grade_many(with_purpose))

          1회  2회  3회  4회  5회
0단계 지시만   10  10  10  10  10
4단계 + 제약  10  11  10  10  11
0단계 지시만의 합계가 높음 0번, 같음 3번, 4단계 + 제약의 합계가 높음 2번
항목별 평균 점수
           0단계 지시만  4단계 + 제약
빠짐없음           3.0       3.0
정확함            2.0       2.4
바로 쓸 수 있음      2.0       2.0
목적 맞춤          3.0       3.0 


검증한 프롬프트를 프롬프트 템플릿으로 정리합니다. 회의록을 `{회의록}` 빈칸으로 바꾸는 일은 코드가 하고, 모델에게는 본문을 바꾸지 말고 옮기게 합니다.

In [ ]:
# 예제 2-9 (프롬프트 템플릿으로 정리합니다)
blank_prompt = prompts['4단계 + 제약'].replace(minutes, '{회의록}')
chat = llm.Chat()
chat.say('아래 프롬프트를 팀원이 다음 주에도 그대로 쓸 수 있게 프롬프트 템플릿 한 장으로 정리해줘.\n'
         '칸: (1) 용도 (2) 넣을 재료와 빼야 할 정보 (3) 프롬프트 본문 (4) 결과 채점 기준 세 줄 (5) 주의할 점 두 줄.\n'
         '(3)은 아래 프롬프트의 문장을 바꾸지 말고 그대로 옮겨.\n\n' + blank_prompt)
bodies = [block for block in re.findall(r'```[^\n]*\n(.*?)```', chat.last.text, flags=re.S) if '{회의록}' in block]
if not bodies:
    print('\n{회의록} 빈칸이 든 코드 블록을 찾지 못했습니다')
else:
    original_lines = blank_prompt.splitlines()
    print('\n코드 블록의 본문이 원래 프롬프트와 같음:', bodies[0].strip() == blank_prompt.strip())
    print('원래 프롬프트에 없는 줄:', [line for line in bodies[0].strip().splitlines() if line not in original_lines] or '없음')

# 프롬프트 템플릿: 회의록 요약 (팀장 보고용)

## (1) 용도
설비 가동률 리스크에 관심이 큰 팀장에게 보고할 회의록 요약본을 만든다. 핵심 요약, 결정사항 표, 액션아이템 목록을 한 번에 뽑는다.

## (2) 넣을 재료와 빼야 할 정보
- **넣을 재료:** 회의록 원문 전체 (일시, 참석자, 안건, 논의 내용, 결정, 액션아이템 포함)
- **빼야 할 정보:** 고객사명과 계약 조건, 공정 레시피나 수율 등 기밀 수치, 개인 평가성 발언, 사외 AI 도구 사용이 금지된 자료. 사내 보안 기준에 맞게 마스킹하거나 삭제한 뒤 넣는다.

## (3) 프롬프트 본문
```
너는 반도체 장비팀 파트장이다.
내일 아침 팀장 보고에 쓸 자료다.
팀장은 설비 가동률에 영향을 주는 리스크를 가장 궁금해한다.
다음 회의록을 요약해줘.

형식:
- 핵심 3줄 요약
- 결정사항 표 (안건 | 결정 | 비고)
- 액션아이템 목록 (담당자와 기한 포함)

제약:
- 회의록에 없는 내용을 추가하지 마
- 담당자나 기한이 불명확하면 '확인 필요'로 표기해
- 존댓말 보고체로 써줘

{회의록}
```

## (4) 결과 채점 기준 (세 줄)
1. 핵심 요약이 정확히 3줄이고, 설비 가동률 관련 리스크가 그중 한 줄 이상에 담겼는가?
2. 결정사항 표와 액션아이템의 내용이 회의록 원문과 일치하고, 원문에 없는 내용이 추가되지 않았는가?
3. 담당자나 기한이 불명확한 항목에 '확인 필요'가 표기되었고, 문장이 존댓말 보고체로 통일되었는가?

## (5) 주의할 점 (두 줄)
- `{회의록}`은 중괄호까지 포함해 실제 회의록 원문으로 통째로 바꿔 넣는다. 이 자리를 비워 두면 AI가 내용을 지어낼 수 있다.
- 보고 전에 담당자, 기한, 숫자를 원문과 직접 대조한다. 회의록에 근거가 없는 항목은 '확인 필요'로 표기됐는지 확인한다.

코드 블록의 본문이 원래 프롬프트와 같음: True
원래 프롬프트에 없는 줄: 없음


첫 채점에서 0단계 요약은 다섯 번 모두 7점이었고 4단계 요약은 6~7점이었습니다. 다섯 번 가운데 0단계가 높은 것이 두 번, 같은 것이 세 번입니다. 요약의 순서를 바꿔 4단계 요약을 A로 보여 준 채점에서도 0단계가 높은 것이 두 번, 같은 것이 세 번이었습니다(4단계 7점, 0단계 7~9점). 순서를 바꿔도 4단계가 더 높게 나온 적은 없으므로 위치 때문에 생긴 차이로 보지 않습니다. 다만 0단계 요약의 점수는 순서를 바꾼 채점에서 7~9점으로 퍼져, 같은 요약도 채점마다 점수가 달라졌습니다. 항목별 평균을 보면 첫 채점에서 빠짐없음은 둘 다 3.0, 정확함은 둘 다 2.0이었고, 4단계 요약이 바로 쓸 수 있음(1.6, 순서를 바꾼 채점에서는 2.0)에서 낮았습니다. 목적 맞춤 항목을 더해 채점하자 방향이 바뀌어, 4단계가 높은 것이 두 번, 같은 것이 세 번이었습니다(합계 10~11점 대 10점). 목적 맞춤은 둘 다 3.0이었으므로 차이는 목적 맞춤이 아니라 정확함의 평균(2.4 대 2.0)에서 났습니다. 같은 요약 두 개를 채점했는데 항목 하나를 더한 채점에서 정확함의 평균이 달라졌으므로, 이 정도 차이로 어느 쪽이 낫다고 결론짓지 않습니다.

점수와 함께 근거를 인용하게 한 결과에서 A(0단계)는 가장 낮은 정확함 2점의 근거로 요약 끝의 "참고로 2라인 파티클 상승은 원인이 불명확해서, 필터 교체 후에도 수치를 계속 확인해야 할 것 같습니다"를 인용했습니다. 회의록에 없는 의견이므로 정확함의 감점과 연결해 읽을 수 있습니다. B(4단계)는 정확함과 바로 쓸 수 있음이 2점이었고, 근거로는 "가동률 관련 리스크는 3건입니다." 한 구절만 인용했습니다. 이 인용만으로는 어느 항목을 왜 감점했는지 알 수 없습니다. 근거는 인용 한 줄이 아니라 어느 항목에 왜 몇 점을 주었는지를 문장으로 적게 해야 합니다.

이 결과로는 4단계 프롬프트가 나아졌다고도 나빠졌다고도 말할 수 없습니다. 이유는 두 가지입니다. 하나는 채점 자체가 실행마다 달라져, 다섯 번의 채점으로는 1점 안팎의 차이가 프롬프트 때문인지 판단할 수 없다는 점입니다. 2026-11-23 실행에서는 첫 채점에서 0단계가 다섯 번 모두 9점으로 4단계(7~8점)보다 높았고, 목적 맞춤을 더해도 순위가 바뀌지 않았습니다. 같은 프롬프트로 만든 요약과 같은 루브릭으로도 실행한 날에 따라 결론의 방향이 달라진 것입니다. 다른 하나는 4단계 프롬프트가 고친 점이 루브릭의 항목에 없다는 점입니다. 루브릭에는 요청한 형식(3줄 요약, 표, 액션아이템)을 지켰는지, 담당 칸에 지어낸 값이 없는지를 보는 항목이 없습니다. 0단계 프롬프트는 형식을 요청하지 않았으므로, 4단계가 형식을 지킨 것은 점수에 반영되지 않습니다. 고친 프롬프트의 점수가 더 낮게 나오면 같은 채점을 여러 번 해 분포를 보고, 순서를 바꿔 채점하고, 루브릭이 프롬프트의 목적과 요구를 항목으로 담고 있는지 확인하고, 마지막으로 사람이 응답을 읽습니다. 채점도 모델이 한 것이므로 채점 결과 자체도 사람이 확인합니다.

프롬프트 템플릿은 다섯 칸으로 정리되었습니다. 응답의 코드 블록 본문은 원래 프롬프트와 같았고, 원래 프롬프트에 없는 줄도 없었습니다. (2)의 "빼야 할 정보", (4)의 채점 기준, (5)의 주의는 원래 프롬프트에 없던 내용을 모델이 만든 것입니다. 팀에 남기는 문서이므로 이 칸들의 내용도 읽고 맞는지 확인합니다.

## 프롬프트 템플릿

> **요점**
>
> 프롬프트 템플릿은 검증한 프롬프트에서 매번 바뀌는 곳만 빈칸으로 남긴 문서입니다. 용도, 재료와 뺄 정보, 본문, 채점 기준, 주의의 다섯 칸을 적어 두면 다음에 프롬프트를 고칠 때 같은 기준으로 다시 비교할 수 있습니다.

프롬프트 템플릿은 팀원이 같은 업무를 같은 방식으로 처리하도록 프롬프트를 남긴 문서입니다. 다섯 칸으로 씁니다.

1. **용도**: 이 프롬프트를 언제 쓰는지 한 줄로 적습니다.
2. **재료**: 넣을 자료와 빼야 할 정보(실명, 미공개 수치, 고객사명)를 적습니다.
3. **프롬프트 본문**: 검증한 문장 그대로 두고, 매번 바뀌는 곳만 `{회의록}` 같은 빈칸으로 만듭니다.
4. **채점 기준**: 루브릭을 함께 적어 두어, 다음에 프롬프트를 고칠 때 같은 기준으로 다시 비교합니다.
5. **주의**: 결과에서 사람이 꼭 대조할 것을 두 줄로 적습니다.

본문의 문장을 바꾸지 않는 이유는 검증한 것이 바로 그 문장이기 때문입니다. 모델이 템플릿을 정리하며 문장을 바꾸거나 빈칸의 위치를 옮길 수 있으므로, 예제 2-9처럼 본문이 그대로인지 코드로 확인합니다.

자주 쓰는 지시 문구를 모았습니다. 앞의 예제에서 쓴 문장을 그대로 또는 줄여서 옮겼습니다.

| 하려는 일 | 지시 문구 |
| --- | --- |
| 없는 내용 줄이기 | `회의록에 없는 내용을 추가하지 마` |
| 비어 있는 칸을 표시하기 | `담당자나 기한이 불명확하면 '확인 필요'로 표기해` |
| 분류 이름을 고정하기 | `반드시 [설비이상/품질/안전/운영] 네 라벨만 사용해줘` |
| 응답을 코드로 읽기 | `다른 말 없이, 코드 블록 표시도 없이 순수 JSON만 출력해` |
| 중간 값을 검산하기 | `단계별로 계산 과정을 보여준 뒤 마지막에 답을 정리해줘` |
| 응답을 검증하기 | `위 답변에서 널리 알려진 사실, 문서로 확인이 필요한 것, 네가 추측한 것을 나눠서 표시해줘` |

## 실습

실습은 `1일차_02_실습.ipynb`에서 합니다. 재료는 야간 설비 알람 로그 10건(`알람로그_10건.txt`)과 분류 기준(`알람_분류기준.txt`)이고, 줄별 정답(`알람로그_정답.txt`)은 테스트 셀이 읽습니다. 두 실습 모두 테스트 셀이 실제 API를 호출해 얻은 응답을 검사하므로 요금이 조금 듭니다. 풀이와 해설은 `1일차_02_실습_풀이.ipynb`에 있습니다.

**실습 2. 알람 로그를 분류하는 프롬프트 작성**

역할 지정, 맥락, 분류 기준, 출력 형식을 넣은 프롬프트로 알람 로그 10건을 조치 필요와 모니터링으로 분류합니다. 예시는 세 번의 분류가 달라질 때 더합니다. 출력은 코드가 읽을 수 있게 한 줄에 하나씩 `번호 | 분류`로만 받습니다.

완료 기준: 테스트 셀이 "실습 2 통과"를 출력합니다. 같은 프롬프트를 새 채팅에서 세 번 보내, 세 번 모두 10건을 읽을 수 있고, 형식 외의 말이 없고, 세 번의 분류가 같고, 정답과 9건 이상 일치해야 합니다.

**실습 3. 분류 결과에서 표를 만들고 개수를 확인하는 체인**

실습 2의 분류 결과를 체인의 첫 단계(1단)로 삼아, 조치 필요인 로그만 한 줄을 한 행으로 정리한 마크다운 표를 만드는 둘째 단계(2단)를 보냅니다. 표의 데이터 줄 수를 세는 `count_table_rows()`를 작성하고, 표에서 센 행 수와 로그 번호가 정답과 같은지 확인합니다. 같은 설비가 두 번 나와도 행을 합치지 않게 하는 문장을 빼고 표가 어떻게 바뀌는지 비교한 뒤, 표를 실습 노트북에 적힌 루브릭 네 항목으로 직접 채점합니다.

완료 기준: 테스트 셀이 "실습 3 통과"를 출력합니다. 표에서 센 행 수가 조치 필요 건수와 같고 로그 번호가 정답의 조치 필요 번호와 같아야 합니다. 1단의 분류가 한 건이라도 틀리면 표를 바르게 만들어도 테스트가 실패하므로, 실습 2의 분류가 정답과 10건 모두 일치한 결과로 진행합니다.

풀이를 실행한 결과는 다음과 같습니다. 여러분의 결과와 비교해 봅니다.

| 확인한 것 | 풀이 실행 결과 (2026-11-23, 소형 모델) |
| --- | --- |
| 실습 2 | 세 번 모두 같은 분류였고 정답과 10건이 모두 일치함. 테스트 4개 통과 |
| 실습 3의 표 | 7행이고 로그 번호는 1, 2, 4, 5, 7, 8, 9. 정답의 조치 필요 건과 같음. 테스트 4개 통과 |
| 합치지 말라는 문장을 뺀 두 번 | 두 번 모두 7행이고 로그 번호도 같음. 이 모델에서는 문장이 없어도 같은 설비의 두 줄을 합치지 않음 |

표의 내용을 읽으면 확인할 곳이 남아 있습니다. 로그 번호 4번의 행에서 알람 칸은 "확인 필요(원자료에 '재발'만 기재)"입니다. 1번 줄을 보면 E-201이 펌프 압력 이상임을 알 수 있는데도 알람 칸을 채우지 않았고, 알람 코드 E-201은 설비 칸에 들어갔습니다. 행 수와 로그 번호는 테스트를 통과했지만, 파트장이 그 행의 알람 칸만 보면 무슨 알람인지 알 수 없습니다. 문장을 뺀 실험에서 행이 합쳐지지 않았다는 결과는 제약이 필요 없다는 뜻이 아닙니다. 모델과 응답에 따라 합쳐질 수 있으므로 제약은 남기고, 행 수는 코드로 확인합니다.

## 핵심 정리

- 프롬프트의 요소는 역할 지정, 맥락, 형식, 제약, 예시의 다섯 가지입니다. 하나씩 더하며 응답의 어느 부분이 바뀌는지 확인해야 필요한 줄만 남깁니다.
- 분류 프롬프트에는 허용 라벨, 라벨마다 예시, 출력 형식을 적습니다. 모델이 적은 확신도는 생성한 글이므로 사람이 먼저 볼 목록을 고르는 참고로만 씁니다.
- 출력 형식은 결과를 사용할 곳에 맞춰 정합니다. 지시한 형식이 지켜졌는지는 코드로 확인하고, 지켜지지 않을 수 있다는 전제로 후속 처리를 설계합니다.
- CoT를 쓰면 중간 값을 검산할 수 있습니다. 여러 단계의 계산이나 조건 판단에 적용하고, 판단이 필요 없는 일에는 적용하지 않습니다.
- 매번 똑같이 적용할 역할, 어조, 형식, 제약은 시스템 프롬프트에 두고, 이번 건의 조건과 원자료는 사용자 메시지에 둡니다. 규칙은 규칙마다 지켜졌는지 확인합니다.
- 긴 문서는 한 번에 넣어 보고, 컨텍스트 윈도우를 넘거나 응답이 잘릴 때만 구간으로 나눕니다. 나누면 모델이 구간마다 따로 판단하므로 같은 지시와 제외 기준을 구간마다 적고, 결과는 정답이나 원문과 대조합니다.
- 프롬프트 체인은 한 단계에 한 가지만 시키고, 중간 산출물을 남기고, 끝에서 개수를 셉니다. 모델이 답한 개수는 코드로 센 개수와 비교하고, 둘이 다르면 코드로 센 개수를 믿습니다.
- 검증은 세 단계입니다. 모델의 자기 검증은 어디부터 확인할지 표시할 뿐이고, 다른 사람에게 전달하는 문서는 숫자와 고유명사를 근거 자료와 대조합니다.
- 프롬프트를 고친 효과는 루브릭으로 같은 기준을 세우고 여러 번 채점해 판단합니다. 검증한 프롬프트는 본문을 바꾸지 않고 프롬프트 템플릿으로 남깁니다.

## 자가점검

**1.** 프롬프트에 "팀장은 설비 가동률에 영향을 주는 리스크를 가장 궁금해한다"를 더했습니다. 이 문장은 다섯 요소 가운데 무엇이고, 응답의 어느 부분을 바꾸기를 기대합니까?

<details><summary>정답 보기</summary>

맥락입니다. 결과를 누가 읽고 무엇을 궁금해하는지 알려 주므로, 어떤 안건을 앞에 둘지와 어떤 내용을 길게 쓸지를 바꾸기를 기대합니다. 바뀌었는지는 응답을 읽거나 코드로 세어 확인합니다.

</details>

**2.** 분류 프롬프트에 허용 라벨, 예시, 출력 형식을 적는 이유를 하나씩 말해 보세요.

<details><summary>정답 보기</summary>

허용 라벨은 분류 이름과 개수가 응답마다 달라지는 것을 막습니다. 예시는 입력에서 정답으로 이어지는 패턴을 보여 주어, 실행마다 라벨이 달라지는 입력의 라벨을 하나로 정해 줍니다. 출력 형식은 코드가 응답을 읽을 수 있게 합니다.

</details>

**3.** Few-shot 예시를 고르는 방법으로 맞는 것은 무엇입니까?

- A. 가장 쉬운 입력 하나를 고른다
- B. 라벨마다 하나 이상, 엣지 케이스를 포함해 다양하게, 실제 업무와 비슷한 것을 고른다
- C. 예시가 많을수록 좋으므로 가능한 한 많이 넣는다

<details><summary>정답 보기</summary>

**B.** Anthropic의 공식 문서도 예시를 3~5개 넣고 엣지 케이스를 포함해 다양하게 고르며 실제 업무와 비슷한 것을 고르라고 안내합니다. 예시가 한 라벨에만 있으면 나머지 라벨의 규칙을 보여 주지 못합니다.

</details>

**4.** 모델이 분류 8건에 모두 확신도 "상"을 적었습니다. 이 확신도를 어떻게 써야 합니까?

<details><summary>정답 보기</summary>

확신도는 모델이 측정한 값이 아니라 생성한 글입니다. 같은 입력을 다시 보내면 달라질 수 있으므로 믿을 수 있는 정도의 증거로 쓰지 않습니다. "하"나 "중"으로 적은 줄을 사람이 먼저 볼 목록으로 고르는 참고로만 쓰고, 모두 "상"이어도 분류를 정답 라벨과 대조해 확인합니다.

</details>

**5.** CoT를 적용하면 좋은 작업을 모두 고르세요. (가) 세 장비의 생산량을 곱하고 더해 총합을 구한다 (나) 영문 회의록을 번역한다 (다) 회의록에서 "금요일"이라는 말이 들어 있는 줄의 번호를 뽑는다 (라) 분류한 근거를 줄마다 적게 한다

<details><summary>정답 보기</summary>

(가)와 (라)입니다. 여러 단계의 계산과 근거가 필요한 판단은 중간 값을 적게 하면 검산할 수 있습니다. (나)와 (다)는 판단이 필요 없는 변환과 추출이라 출력 토큰과 시간만 늘어납니다.

</details>

**6.** 다음 네 가지를 시스템 프롬프트와 사용자 메시지로 나눠 보세요. (가) 존댓말 보고체로 쓴다 (나) 이번 주 회의록 전문 (다) 원자료에 없으면 확인 필요로 적는다 (라) 이번 건은 금요일 마감이다

<details><summary>정답 보기</summary>

(가)와 (다)는 매번 똑같이 적용되므로 시스템 프롬프트에 둡니다. (나)와 (라)는 이번 건에만 해당하고 자주 바뀌므로 사용자 메시지에 둡니다.

</details>

**7.** 4주치 교대 일지 400줄에서 안전 항목을 뽑으려 합니다. 어떤 순서로 진행합니까?

<details><summary>정답 보기</summary>

먼저 한 번에 넣어 봅니다. 입력이 들어가고 응답이 잘리지 않으면 나누지 않습니다. 컨텍스트 윈도우를 넘거나 응답이 잘리면 구간으로 나누되 구간마다 같은 지시와 제외 기준을 적습니다. 합친 뒤에는 구간별 결과의 합과 합친 항목 수를 비교하고, 넣을지 뺄지 애매한 줄은 원문과 대조합니다.

</details>

**8.** 체인의 마지막 단계에서 항목 수를 모델에게 세게 하지 않고 코드로 세는 이유는 무엇입니까?

<details><summary>정답 보기</summary>

항목 수는 코드가 정확하게 셀 수 있는 값이고, 모델이 센 값이 맞았는지는 응답만으로 알 수 없습니다(파트 1의 계산의 한계와 같은 원칙입니다). 코드로 센 값과 모델이 답한 값을 나란히 놓으면 둘이 어긋나는 경우를 바로 찾을 수 있습니다. 개수가 맞아도 틀린 줄은 찾아낼 수 없으므로 원문 대조는 여전히 필요합니다.

</details>

**9.** 자기 검증에서 모델이 "널리 알려진 사실"로 분류한 문장은 확인하지 않아도 됩니까?

<details><summary>정답 보기</summary>

확인해야 합니다. 모델은 틀린 내용을 틀린 줄 모르고 사실로 분류할 수 있습니다. 자기 검증은 어디부터 확인할지 표시해 줄 뿐입니다. 다른 사람에게 전달하는 문서에서는 분류와 관계없이 숫자와 고유명사를 원문과 대조합니다.

</details>

**10.** 프롬프트를 고쳤더니 루브릭 점수가 고치기 전보다 낮게 나왔습니다. 프롬프트가 나빠졌다고 결론짓기 전에 확인할 것 두 가지는 무엇입니까?

<details><summary>정답 보기</summary>

첫째, 같은 채점을 여러 번 해서 점수의 분포를 봅니다. 모델의 채점은 실행마다 달라지므로 한 번의 순위로 결론짓지 않습니다. 둘째, 루브릭이 프롬프트의 목적을 담고 있는지 봅니다. 고친 목적이 항목에 없으면 고친 효과가 점수에 나타나지 않거나 오히려 감점될 수 있습니다. 마지막으로 사람이 응답을 읽고 확인합니다.

</details>